# Amazon ML Challenge 2026: Business Entity Resolution
## Candidate-Pair Matching Model Training & Validation (Google Colab Ready)

**NOTE: DEVELOPMENT/TRAINING EXPERIMENT** — 150k S1 entities + random distractors. Not final full-scale training.

### Pipeline Architecture:
1. Environment Setup
2. S1 Sampling & Train/Val Split (80/20, zero leakage)
3. 8-Pass Blocking (K=25)
4. Deliberate Hard-Negative Sampling (6 categories)
5. Feature Engineering (24 features)
6. Model Training: LR vs XGBoost
7. Threshold Search & Macro F0.5
8. Error Analysis
9. Artifact Export


### Section 1: Environment Setup & Reproducibility


In [ ]:
import os, sys, time, json, random, platform
import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print('=' * 60)
print(f'Python:  {platform.python_version()}')
print(f'OS:      {platform.system()} {platform.release()}')
print(f'CPUs:    {os.cpu_count()}')
print(f'NumPy:   {np.__version__}')
print(f'Pandas:  {pd.__version__}')
print(f'SEED:    {SEED}')
print('=' * 60)


### Section 2: Dataset Location — ZIP Upload (/content)

The dataset ZIP is uploaded directly to `/content` in the Colab runtime.
This cell automatically detects the ZIP, extracts it, locates the four required
TSV files, and sets `DATA_ROOT`. No Google Drive is used.


In [ ]:
import glob, zipfile

EXTRACT_DIR   = '/content/amazon_ml_challenge'
REQUIRED_FILES = [
    'train_source1.tsv',
    'train_source2.tsv',
    'train_source3.tsv',
    'train_ground_truth.tsv',
]

# ── 1. Locate the uploaded ZIP under /content ──────────────────────────────
zip_candidates = glob.glob('/content/*.zip')
if not zip_candidates:
    raise FileNotFoundError(
        'No ZIP file found in /content. '
        'Please upload the dataset ZIP directly to the Colab runtime.'
    )
if len(zip_candidates) > 1:
    raise FileNotFoundError(
        f'Multiple ZIP files found in /content: {zip_candidates}. '
        'Please ensure only the dataset ZIP is present.'
    )
zip_path = zip_candidates[0]
print(f'ZIP found:            {zip_path}')
print(f'Extraction directory: {EXTRACT_DIR}')

# ── 2. Extract ZIP to /content/amazon_ml_challenge ────────────────────────
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

# ── 3. Recursively locate all four required TSV files ─────────────────────
found = {}
for root, dirs, files in os.walk(EXTRACT_DIR):
    for fname in files:
        if fname in REQUIRED_FILES and fname not in found:
            found[fname] = os.path.join(root, fname)

missing = [f for f in REQUIRED_FILES if f not in found]
if missing:
    raise FileNotFoundError(
        f'Could not locate the following files after extraction: {missing}\n'
        f'Files found so far: {list(found.keys())}\n'
        f'Check the ZIP structure under {EXTRACT_DIR}'
    )

# ── 4. Set DATA_ROOT to the directory containing the four files ───────────
# All four files must be in the same directory
dirs_found = {os.path.dirname(p) for p in found.values()}
if len(dirs_found) != 1:
    raise FileNotFoundError(
        f'Required TSV files are spread across multiple directories: {dirs_found}. '
        'Expected all four files in the same folder.'
    )
DATA_ROOT = dirs_found.pop()
print(f'Resolved DATA_ROOT:   {DATA_ROOT}')

# ── 5. Print each file with full path and size ────────────────────────────
print()
print(f'  {"File":<28} {"Size":>10}  Path')
print(f'  {"-"*28} {"-"*10}  {"-"*40}')
for fname in REQUIRED_FILES:
    fpath = found[fname]
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f'  {fname:<28} {size_mb:>8.1f}MB  {fpath}')

# ── 6. Final verification ─────────────────────────────────────────────────
for fname in REQUIRED_FILES:
    if not os.path.isfile(found[fname]):
        raise FileNotFoundError(f'Verification failed: {found[fname]} is not a file.')
print()
print('All 4 required training files verified. DATA_ROOT is set.')
print('Ready to proceed — datasets will be loaded in Section 4.')


### Section 3: Install Dependencies


In [ ]:
!pip install -q duckdb>=1.1.0 rapidfuzz>=3.0.0 xgboost>=2.0.0 scikit-learn>=1.3.0 joblib>=1.3.0

import duckdb, rapidfuzz, xgboost as xgb, sklearn
print(f'DuckDB:{duckdb.__version__}  RapidFuzz:{rapidfuzz.__version__}  XGBoost:{xgb.__version__}  sklearn:{sklearn.__version__}')


### Section 4: Load and Validate Data


In [ ]:
s1_path = os.path.join(DATA_ROOT,'train_source1.tsv').replace('\\','/')
s2_path = os.path.join(DATA_ROOT,'train_source2.tsv').replace('\\','/')
s3_path = os.path.join(DATA_ROOT,'train_source3.tsv').replace('\\','/')
gt_path = os.path.join(DATA_ROOT,'train_ground_truth.tsv').replace('\\','/')

DB_PATH = '/content/duckdb_training.db'
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
con = duckdb.connect(DB_PATH)
con.execute("SET memory_limit='4GB';")
con.execute('SET threads=4;')
con.execute("SET temp_directory='/content/duckdb_tmp';")
os.makedirs('/content/duckdb_tmp', exist_ok=True)

for name, p in [('Source 1',s1_path),('Source 2',s2_path),('Source 3',s3_path),('Ground Truth',gt_path)]:
    df = con.execute(f"SELECT * FROM read_csv_auto('{p}', delim='\t', header=true, quote='') LIMIT 3").df()
    cnt = con.execute(f"SELECT count(*) FROM read_csv_auto('{p}', delim='\t', header=true, quote='')").fetchone()[0]
    print(f'\n--- {name} ({cnt:,} rows) ---')
    print(f'Columns: {list(df.columns)}')
    display(df)


### Section 5: Preprocessing & SQL Normalization Macros


In [ ]:
con.execute("""
CREATE OR REPLACE MACRO norm_text(s) AS
    lower(trim(regexp_replace(regexp_replace(
        regexp_replace(coalesce(s,''), 'https?://(?:www\.)?|www\.', '', 'g'),
        '\.(?:com|org|net|in|co|gov|edu|fr|io)\\b', ' ', 'g'),
        '[^\\w\\s]', ' ', 'g')));

CREATE OR REPLACE MACRO clean_legal(s) AS
    trim(regexp_replace(norm_text(s),
    '\\b(?:private\\s+limited|pvt\\s+ltd|pvt\\s+limited|incorporated|corporation|enterprises|enterprise|associates|industries|industry|holdings|holding|services|service|company|limited|llc|inc|corp|ltd|llp|co|sarl|sas|sa|sci|eurl|sasu)\\b',
    ' ', 'g'));

CREATE OR REPLACE MACRO compact_name(s) AS
    regexp_replace(norm_text(s), '\\s+', '', 'g');

CREATE OR REPLACE MACRO get_token(s, n) AS
    CASE WHEN array_length(string_split(trim(coalesce(s,'')), ' ')) >= n
         THEN string_split(trim(coalesce(s,'')), ' ')[n]
         ELSE '' END;

CREATE OR REPLACE MACRO clean_addr(s) AS
    lower(trim(regexp_replace(
        regexp_replace(coalesce(s,''), '[^\\w\\s]', ' ', 'g'),
        '\\s+', ' ', 'g')));
""")
print('DuckDB preprocessing macros registered.')


### Section 6: S1 Entity Sampling & Train/Validation Split

**Anti-Leakage Guarantee**: All candidate pairs for a given S1 entity stay entirely in train OR val.
Explicit leakage checks are printed and will raise an error if any leakage is detected.


In [ ]:
TRAIN_S1_LIMIT = 150000
VALIDATION_FRACTION = 0.20
CHUNK_SIZE = 50000
TOP_K = 25
HARD_NEG_RATIO = 3.0

print(f'Sampling {TRAIN_S1_LIMIT} S1 entities with SEED={SEED}...')
sample_clause = f'USING SAMPLE {TRAIN_S1_LIMIT} (reservoir, {SEED})'

con.execute(f"""
CREATE OR REPLACE TABLE s1_sample AS
SELECT entity_id, coalesce(country,'') as country,
    business_name as raw_name, business_address as raw_addr,
    norm_text(business_name) as norm_name,
    clean_legal(business_name) as core_name,
    clean_addr(business_address) as norm_addr,
    get_token(clean_legal(business_name),1) as p1,
    get_token(clean_legal(business_name),2) as p2,
    get_token(clean_legal(business_name),3) as p3,
    left(compact_name(business_name),4) as cmp4,
    left(compact_name(business_name),5) as cmp5,
    regexp_extract(clean_addr(business_address),'\\b(\\d{{3,}})\\b',1) as postal,
    regexp_extract(clean_addr(business_address),'^(\\d+)',1) as door,
    get_token(clean_addr(business_address),1) as a1,
    get_token(clean_addr(business_address),2) as a2
FROM read_csv_auto('{s1_path}', delim='\t', header=true, quote='')
{sample_clause};
""")

con.execute(f"""
CREATE OR REPLACE TABLE s1_gt AS
SELECT s.*, g.matched_entity_ids,
    case when g.matched_entity_ids is null or trim(g.matched_entity_ids)='' then 0
         else length(g.matched_entity_ids)-length(replace(g.matched_entity_ids,',',''))+1 end as match_count,
    case when g.matched_entity_ids is null or trim(g.matched_entity_ids)='' then 1 else 0 end as is_singleton
FROM s1_sample s
LEFT JOIN read_csv_auto('{gt_path}', delim='\t', header=true, quote='') g
ON s.entity_id = g.source1_entity_id;
""")

con.execute(f"""
CREATE OR REPLACE TABLE s1_split AS
SELECT *,
    case when row_number() over (partition by country,is_singleton order by hash(entity_id))
              <= count(*) over (partition by country,is_singleton) * (1.0 - {VALIDATION_FRACTION})
         then 'train' else 'val' end as split
FROM s1_gt;
""")

dist_df = con.execute("""
SELECT split, country, count(*) as s1_count,
       sum(is_singleton) as singleton_count,
       round(sum(is_singleton)::float/count(*)*100,2) as singleton_pct,
       sum(match_count) as total_true_matches,
       round(avg(match_count),2) as avg_matches_per_s1
FROM s1_split GROUP BY split,country ORDER BY split,country
""").df()
print('\nTrain / Validation S1 Entity Distributions:')
display(dist_df)

# ── EXPLICIT LEAKAGE CHECKS ────────────────────────────────────────────────
print('\n' + '='*60)
print('LEAKAGE CHECKS')
print('='*60)

# Check 1: no S1 entity appears in both splits
overlap = con.execute("""
    SELECT count(*) FROM
        (SELECT entity_id FROM s1_split WHERE split='train') t
        JOIN (SELECT entity_id FROM s1_split WHERE split='val') v
        ON t.entity_id = v.entity_id
""").fetchone()[0]
print(f'S1 entities in BOTH train and val: {overlap}')
if overlap > 0:
    raise AssertionError(f'LEAKAGE DETECTED: {overlap} S1 entities appear in both splits!')
print('  [PASS] No S1 entity appears in both splits.')

# Check 2: every s1_split row has exactly one split value
multi_split = con.execute("""
    SELECT count(*) FROM (
        SELECT entity_id, count(distinct split) as n
        FROM s1_split GROUP BY entity_id HAVING n > 1
    )
""").fetchone()[0]
print(f'S1 entities with multiple split assignments: {multi_split}')
if multi_split > 0:
    raise AssertionError(f'LEAKAGE DETECTED: {multi_split} S1 entities have multiple split values!')
print('  [PASS] Every S1 entity has exactly one split assignment.')

# Check 3: split counts sum to total
total_s1 = con.execute('SELECT count(*) FROM s1_split').fetchone()[0]
train_s1 = con.execute("SELECT count(*) FROM s1_split WHERE split='train'").fetchone()[0]
val_s1   = con.execute("SELECT count(*) FROM s1_split WHERE split='val'").fetchone()[0]
print(f'Total S1: {total_s1:,}  Train: {train_s1:,}  Val: {val_s1:,}  Sum: {train_s1+val_s1:,}')
if train_s1 + val_s1 != total_s1:
    raise AssertionError('LEAKAGE DETECTED: split counts do not sum to total!')
actual_val_frac = val_s1 / total_s1
print(f'  [PASS] Val fraction: {actual_val_frac:.3f} (target {VALIDATION_FRACTION})')
print('All leakage checks passed.')


### Section 7: Final 8-Pass Blocking & Chunked Candidate Ranking

**Development Experiment**: true targets + random S2/S3 distractors (150k each).
Target pool is deduplicated to one unique record per entity_id before blocking.

**Memory Architecture**: Raw candidate pairs (~3.25M) are generated once and stored
on disk. Jaccard scoring + top-K ranking is then done in **S1 chunks of 10,000**,
writing only the top-25 rows per S1 to the persistent `topk_candidates` table.
The full scored intermediate is never materialized in memory.

> **Validation Scope**: This model validation measures matcher performance **conditional on the true target being present in the candidate-generation pool**. End-to-end blocking recall is evaluated separately by `blocking_experiment.py`. The F0.5 reported here is **not** the final end-to-end competition score.


In [ ]:
print('Building Target Pool (true targets + random distractors)...')
t0_pool = time.time()

con.execute("""
CREATE OR REPLACE TABLE true_target_ids AS
SELECT DISTINCT unnest(string_split(matched_entity_ids,',')) as target_id
FROM s1_gt WHERE matched_entity_ids is not null and trim(matched_entity_ids)!='';
""")

# Build raw pool with UNION ALL (may contain duplicate entity_ids)
con.execute(f"""
CREATE OR REPLACE TABLE target_pool_raw AS
SELECT entity_id, coalesce(country,'') as country,
    business_name as raw_name, business_address as raw_addr,
    norm_text(business_name) as norm_name,
    clean_legal(business_name) as core_name,
    clean_addr(business_address) as norm_addr,
    get_token(clean_legal(business_name),1) as p1,
    get_token(clean_legal(business_name),2) as p2,
    get_token(clean_legal(business_name),3) as p3,
    left(compact_name(business_name),4) as cmp4,
    left(compact_name(business_name),5) as cmp5,
    regexp_extract(clean_addr(business_address),'\\b(\\d{{3,}})\\b',1) as postal,
    regexp_extract(clean_addr(business_address),'^(\\d+)',1) as door,
    get_token(clean_addr(business_address),1) as a1,
    get_token(clean_addr(business_address),2) as a2
FROM (
    SELECT * FROM read_csv_auto('{s2_path}', delim='\t', header=true, quote='')
    WHERE entity_id IN (SELECT target_id FROM true_target_ids)
    UNION ALL
    SELECT * FROM read_csv_auto('{s3_path}', delim='\t', header=true, quote='')
    WHERE entity_id IN (SELECT target_id FROM true_target_ids)
    UNION ALL
    (SELECT * FROM read_csv_auto('{s2_path}', delim='\t', header=true, quote='') USING SAMPLE 150000 (reservoir, {SEED}))
    UNION ALL
    (SELECT * FROM read_csv_auto('{s3_path}', delim='\t', header=true, quote='') USING SAMPLE 150000 (reservoir, {SEED}))
);
""")

raw_pool_cnt = con.execute('SELECT count(*) FROM target_pool_raw').fetchone()[0]
dup_cnt_before = con.execute("""
    SELECT count(*) FROM (
        SELECT entity_id FROM target_pool_raw
        GROUP BY entity_id HAVING count(*) > 1
    )
""").fetchone()[0]
print(f'Raw pool (before dedup): {raw_pool_cnt:,} records, {dup_cnt_before:,} entity_ids with duplicates')

# Deduplicate: keep one record per entity_id, preferring true-target rows
con.execute("""
CREATE OR REPLACE TABLE target_pool AS
SELECT entity_id, country, raw_name, raw_addr, norm_name, core_name, norm_addr,
       p1, p2, p3, cmp4, cmp5, postal, door, a1, a2
FROM (
    SELECT *,
        row_number() over (
            partition by entity_id
            -- true-target rows first (entity_id IN true_target_ids = 1),
            -- then deterministic tie-break by entity_id
            order by
                CASE WHEN entity_id IN (SELECT target_id FROM true_target_ids)
                     THEN 0 ELSE 1 END ASC,
                entity_id ASC
        ) as rn
    FROM target_pool_raw
)
WHERE rn = 1;
""")

# ── DUPLICATE CHECK — FAIL LOUDLY IF ANY REMAIN ───────────────────────────
dup_cnt_after = con.execute("""
    SELECT count(*) FROM (
        SELECT entity_id, count(*) as n
        FROM target_pool
        GROUP BY entity_id HAVING n > 1
    )
""").fetchone()[0]
if dup_cnt_after > 0:
    raise AssertionError(f'DEDUP FAILED: {dup_cnt_after} entity_ids still have duplicate records in target_pool!')

# ── TARGET POOL COMPOSITION REPORT ────────────────────────────────────────
true_tgt_cnt = con.execute('SELECT count(*) FROM true_target_ids').fetchone()[0]
total_pool   = con.execute('SELECT count(*) FROM target_pool').fetchone()[0]
true_in_pool = con.execute("""
    SELECT count(*) FROM target_pool
    WHERE entity_id IN (SELECT target_id FROM true_target_ids)
""").fetchone()[0]
rand_distractor_cnt = total_pool - true_in_pool
pct_true = true_in_pool / total_pool * 100 if total_pool > 0 else 0

print('\n' + '='*60)
print('TARGET POOL COMPOSITION  [DEVELOPMENT EXPERIMENT]')
print('='*60)
print(f'  Duplicate entity_ids before dedup:   {dup_cnt_before:>10,}')
print(f'  Duplicate entity_ids after dedup:    {dup_cnt_after:>10,}  [PASS]')
print(f'  True target IDs (unique):            {true_tgt_cnt:>10,}')
print(f'  True target records in pool:         {true_in_pool:>10,}')
print(f'  Random S2+S3 distractor records:     {rand_distractor_cnt:>10,}')
print(f'  Total unique target pool records:    {total_pool:>10,}')
print(f'  True-target % of pool:               {pct_true:>9.2f}%')
print('  NOTE: Dev experiment — full-scale training uses complete S2+S3 corpus.')
print(f'  Built in {time.time()-t0_pool:.2f}s')

# ── 8-PASS BLOCKING ───────────────────────────────────────────────────────
print('\nExecuting 8-pass blocking...')
t_block = time.time()

con.execute("""
CREATE OR REPLACE TABLE raw_candidate_pairs AS
SELECT sub.entity_id as s1_id, sub.target_id, count(*) as pass_count
FROM (
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 AND s.p2=t.p2 WHERE length(s.p1)>=3 AND length(s.p2)>=3
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 AND s.postal=t.postal WHERE length(s.p1)>=3 AND s.postal!=''
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 AND s.door=t.door WHERE length(s.p1)>=3 AND s.door!=''
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.cmp5=t.cmp5 WHERE length(s.cmp5)>=5
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 WHERE length(s.p1)>=5
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.postal=t.postal AND s.door=t.door WHERE s.postal!='' AND length(s.door)>=2
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.door=t.door AND s.a1=t.a1 WHERE length(s.door)>=2 AND length(s.a1)>=4
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.door=t.door AND s.a2=t.a2 WHERE length(s.door)>=2 AND length(s.a2)>=4
) sub
GROUP BY sub.entity_id, sub.target_id;
""")

raw_pairs = con.execute('SELECT count(*) FROM raw_candidate_pairs').fetchone()[0]
print(f'Generated {raw_pairs:,} distinct raw candidate pairs in {time.time()-t_block:.2f}s.')

# ── CHUNKED TOP-K RANKING ─────────────────────────────────────────────────
# Score and rank candidates in S1 chunks of RANK_CHUNK_SIZE.
# Only top-K rows per S1 are written to topk_candidates.
# The full scored intermediate is never materialized in memory.
RANK_CHUNK_SIZE = 10000

print(f'Chunked ranking: K={TOP_K}, chunk_size={RANK_CHUNK_SIZE}...')
t_rank = time.time()

# Fetch the ordered list of distinct S1 IDs from raw_candidate_pairs
s1_ids_all = [r[0] for r in con.execute(
    'SELECT DISTINCT s1_id FROM raw_candidate_pairs ORDER BY s1_id'
).fetchall()]
n_s1 = len(s1_ids_all)
print(f'  S1 entities with raw candidates: {n_s1:,}')

# Create the persistent topk_candidates table (empty, correct schema)
con.execute("""
CREATE OR REPLACE TABLE topk_candidates (
    s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,
    split VARCHAR, s1_country VARCHAR,
    s1_raw_name VARCHAR, s1_raw_addr VARCHAR,
    s1_norm_name VARCHAR, s1_norm_addr VARCHAR,
    s1_postal VARCHAR, s1_door VARCHAR,
    t_country VARCHAR,
    t_raw_name VARCHAR, t_raw_addr VARCHAR,
    t_norm_name VARCHAR, t_norm_addr VARCHAR,
    t_postal VARCHAR, t_door VARCHAR,
    jaccard_score DOUBLE, candidate_rank BIGINT
);
""")

n_chunks = (n_s1 + RANK_CHUNK_SIZE - 1) // RANK_CHUNK_SIZE
total_written = 0

for chunk_idx in range(n_chunks):
    chunk_s1 = s1_ids_all[chunk_idx * RANK_CHUNK_SIZE : (chunk_idx + 1) * RANK_CHUNK_SIZE]
    con.register('chunk_s1_ids', __import__('pandas').DataFrame({'s1_id': chunk_s1}))

    con.execute(f"""
    INSERT INTO topk_candidates
    SELECT s1_id, target_id, pass_count,
           split, s1_country,
           s1_raw_name, s1_raw_addr,
           s1_norm_name, s1_norm_addr,
           s1_postal, s1_door,
           t_country,
           t_raw_name, t_raw_addr,
           t_norm_name, t_norm_addr,
           t_postal, t_door,
           jaccard_score, candidate_rank
    FROM (
        SELECT
            c.s1_id, c.target_id, c.pass_count,
            s.split, s.country AS s1_country,
            s.raw_name AS s1_raw_name, s.raw_addr AS s1_raw_addr,
            s.norm_name AS s1_norm_name, s.norm_addr AS s1_norm_addr,
            s.postal AS s1_postal, s.door AS s1_door,
            t.country AS t_country,
            t.raw_name AS t_raw_name, t.raw_addr AS t_raw_addr,
            t.norm_name AS t_norm_name, t.norm_addr AS t_norm_addr,
            t.postal AS t_postal, t.door AS t_door,
            (0.6 * (len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float /
                    nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0)) +
             0.4 * (len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float /
                    nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0)))
                AS jaccard_score,
            row_number() OVER (
                PARTITION BY c.s1_id
                ORDER BY (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float /
                               nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0)) +
                          0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float /
                               nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0))) DESC,
                c.pass_count DESC
            ) AS candidate_rank
        FROM raw_candidate_pairs c
        JOIN chunk_s1_ids        x ON c.s1_id = x.s1_id
        JOIN s1_split            s ON c.s1_id = s.entity_id
        JOIN target_pool         t ON c.target_id = t.entity_id
    ) ranked
    WHERE candidate_rank <= {TOP_K};
    """)

    chunk_written = con.execute('SELECT count(*) FROM topk_candidates').fetchone()[0] - total_written
    total_written += chunk_written
    if (chunk_idx + 1) % 5 == 0 or chunk_idx == n_chunks - 1:
        print(f'  Chunk {chunk_idx+1:>3}/{n_chunks}  '
              f'S1s processed: {min((chunk_idx+1)*RANK_CHUNK_SIZE, n_s1):>7,}  '
              f'Rows written so far: {total_written:>9,}')

topk_cnt = con.execute('SELECT count(*) FROM topk_candidates').fetchone()[0]
per_s1 = [r[0] for r in con.execute('SELECT count(*) FROM topk_candidates GROUP BY s1_id').fetchall()]
per_s1 = [x for x in per_s1 if x > 0]
rank_elapsed = time.time() - t_rank

print(f'\nChunked Ranking Complete in {rank_elapsed:.1f}s:')
print(f'  Total Retained Pairs:    {topk_cnt:,} across {len(per_s1):,} S1 entities')
print(f'  Max possible (K*S1):     {TOP_K * n_s1:,}')
print(f'  Avg Candidates/S1:       {sum(per_s1)/len(per_s1):.1f}')
print(f'  Median:                  {sorted(per_s1)[len(per_s1)//2]}')
print(f'  Maximum:                 {max(per_s1)}')
if topk_cnt > TOP_K * n_s1:
    raise AssertionError(f'RANKING ERROR: {topk_cnt} rows > theoretical max {TOP_K * n_s1}')
print('  [PASS] Row count within theoretical maximum.')


### Section 7b: Smoke Test — Chunked Ranking Validation (TEST_S1 = 10,000)

Before running the full 150k experiment, validate the chunked ranking architecture:
- Raw candidate count for 10k S1 subset
- Top-K candidate count (must be <= 10,000 x 25 = 250,000)
- Average candidates per S1
- No duplicate (s1_id, target_id) pairs in topk_candidates
- Every retained candidate came from raw_candidate_pairs

**Run this cell first. Only proceed to Section 8 after all checks pass.**


In [ ]:
TEST_S1 = 10000
print('=' * 60)
print(f'SMOKE TEST: Chunked Ranking on {TEST_S1:,} S1 entities')
print('=' * 60)

# Sample TEST_S1 S1 IDs from the full s1_split
test_s1_ids = [r[0] for r in con.execute(f"""
    SELECT entity_id FROM s1_split
    USING SAMPLE {TEST_S1} (reservoir, {SEED})
""").fetchall()]
print(f'Test S1 sample size: {len(test_s1_ids):,}')

# Count raw candidates for this subset
con.register('test_s1_ids_df', __import__('pandas').DataFrame({'s1_id': test_s1_ids}))
test_raw_cnt = con.execute("""
    SELECT count(*) FROM raw_candidate_pairs
    WHERE s1_id IN (SELECT s1_id FROM test_s1_ids_df)
""").fetchone()[0]
print(f'Raw candidates for test subset: {test_raw_cnt:,}')
print(f'Avg raw candidates/S1: {test_raw_cnt/len(test_s1_ids):.1f}')

# Run chunked ranking on test subset only
con.execute("""
CREATE OR REPLACE TABLE topk_smoke_test (
    s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,
    split VARCHAR, s1_country VARCHAR,
    s1_raw_name VARCHAR, s1_raw_addr VARCHAR,
    s1_norm_name VARCHAR, s1_norm_addr VARCHAR,
    s1_postal VARCHAR, s1_door VARCHAR,
    t_country VARCHAR,
    t_raw_name VARCHAR, t_raw_addr VARCHAR,
    t_norm_name VARCHAR, t_norm_addr VARCHAR,
    t_postal VARCHAR, t_door VARCHAR,
    jaccard_score DOUBLE, candidate_rank BIGINT
);
""")

SMOKE_CHUNK = 10000
smoke_chunks = (len(test_s1_ids) + SMOKE_CHUNK - 1) // SMOKE_CHUNK
smoke_written = 0
import pandas as _pd

for ci in range(smoke_chunks):
    chunk = test_s1_ids[ci * SMOKE_CHUNK : (ci + 1) * SMOKE_CHUNK]
    con.register('smoke_chunk_ids', _pd.DataFrame({'s1_id': chunk}))
    con.execute(f"""
    INSERT INTO topk_smoke_test
    SELECT s1_id, target_id, pass_count,
           split, s1_country,
           s1_raw_name, s1_raw_addr,
           s1_norm_name, s1_norm_addr,
           s1_postal, s1_door,
           t_country,
           t_raw_name, t_raw_addr,
           t_norm_name, t_norm_addr,
           t_postal, t_door,
           jaccard_score, candidate_rank
    FROM (
        SELECT
            c.s1_id, c.target_id, c.pass_count,
            s.split, s.country AS s1_country,
            s.raw_name AS s1_raw_name, s.raw_addr AS s1_raw_addr,
            s.norm_name AS s1_norm_name, s.norm_addr AS s1_norm_addr,
            s.postal AS s1_postal, s.door AS s1_door,
            t.country AS t_country,
            t.raw_name AS t_raw_name, t.raw_addr AS t_raw_addr,
            t.norm_name AS t_norm_name, t.norm_addr AS t_norm_addr,
            t.postal AS t_postal, t.door AS t_door,
            (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float /
                  nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0)) +
             0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float /
                  nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0)))
                AS jaccard_score,
            row_number() OVER (
                PARTITION BY c.s1_id
                ORDER BY (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float /
                               nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0)) +
                          0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float /
                               nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0))) DESC,
                c.pass_count DESC
            ) AS candidate_rank
        FROM raw_candidate_pairs c
        JOIN smoke_chunk_ids     x ON c.s1_id = x.s1_id
        JOIN s1_split            s ON c.s1_id = s.entity_id
        JOIN target_pool         t ON c.target_id = t.entity_id
    ) ranked
    WHERE candidate_rank <= {TOP_K};
    """)
    smoke_written = con.execute('SELECT count(*) FROM topk_smoke_test').fetchone()[0]

# ── SMOKE TEST CHECKS ─────────────────────────────────────────────────────
print('\nSmoke Test Checks:')
all_ok = True

# 1. Row count within theoretical max
max_possible = TOP_K * len(test_s1_ids)
ok1 = smoke_written <= max_possible
print(f'  [{"PASS" if ok1 else "FAIL"}] Row count: {smoke_written:,} <= max {max_possible:,}')
if not ok1: all_ok = False

# 2. No duplicate (s1_id, target_id) pairs
dup_pairs = con.execute("""
    SELECT count(*) FROM (
        SELECT s1_id, target_id, count(*) as n
        FROM topk_smoke_test GROUP BY s1_id, target_id HAVING n > 1
    )
""").fetchone()[0]
ok2 = dup_pairs == 0
print(f'  [{"PASS" if ok2 else "FAIL"}] Duplicate (s1_id, target_id) pairs: {dup_pairs}')
if not ok2: all_ok = False

# 3. Every retained candidate came from raw_candidate_pairs
orphan_cnt = con.execute("""
    SELECT count(*) FROM topk_smoke_test tk
    WHERE NOT EXISTS (
        SELECT 1 FROM raw_candidate_pairs r
        WHERE r.s1_id = tk.s1_id AND r.target_id = tk.target_id
    )
""").fetchone()[0]
ok3 = orphan_cnt == 0
print(f'  [{"PASS" if ok3 else "FAIL"}] Candidates not in raw pairs (orphans): {orphan_cnt}')
if not ok3: all_ok = False

# 4. candidate_rank <= TOP_K for all rows
over_k = con.execute(f'SELECT count(*) FROM topk_smoke_test WHERE candidate_rank > {TOP_K}').fetchone()[0]
ok4 = over_k == 0
print(f'  [{"PASS" if ok4 else "FAIL"}] Rows with candidate_rank > {TOP_K}: {over_k}')
if not ok4: all_ok = False

# 5. Per-S1 stats
smoke_per_s1 = [r[0] for r in con.execute('SELECT count(*) FROM topk_smoke_test GROUP BY s1_id').fetchall()]
print(f'  Avg candidates/S1: {sum(smoke_per_s1)/len(smoke_per_s1):.1f}')
print(f'  Max candidates/S1: {max(smoke_per_s1)}')

if not all_ok:
    raise AssertionError('SMOKE TEST FAILED. Fix chunked ranking before running full experiment.')
print('\nAll smoke test checks passed. Proceed to full 150k experiment (Section 8+).')
con.execute('DROP TABLE IF EXISTS topk_smoke_test;')


### Section 8 & 9: Positive / Negative Labels & Deliberate Hard-Negative Sampling

Negatives are sampled across **6 deliberate categories**:
1. `high_name_low_addr` — name sim >= 80, addr sim < 40
2. `high_addr_low_name` — addr sim >= 75, name sim < 40
3. `high_name_conflict_door` — name sim >= 70, door numbers differ
4. `high_addr_diff_name` — addr sim >= 70, name sim < 60
5. `generic_name_collision` — very short/common name (<=5 chars stripped)
6. `remaining_difficult` — high jaccard but uncategorised

Budget is split proportionally across categories. Total = positives x HARD_NEG_RATIO.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE labeled_topk AS
SELECT c.*,
    case when g.target_id is not null then 1 else 0 end as label
FROM topk_candidates c
LEFT JOIN (
    SELECT entity_id as s1_id, unnest(string_split(matched_entity_ids,',')) as target_id
    FROM s1_gt WHERE matched_entity_ids is not null
) g ON c.s1_id=g.s1_id AND c.target_id=g.target_id;
""")

label_dist = con.execute("""
    SELECT split, label, count(*) as pair_count
    FROM labeled_topk GROUP BY split,label ORDER BY split,label
""").df()
print('Raw Candidate Label Distribution:')
display(label_dist)

train_pos_cnt   = con.execute("SELECT count(*) FROM labeled_topk WHERE split='train' AND label=1").fetchone()[0]
total_neg_pool  = con.execute("SELECT count(*) FROM labeled_topk WHERE split='train' AND label=0").fetchone()[0]
max_train_negs  = int(train_pos_cnt * HARD_NEG_RATIO)
HARD_NEG_CHUNK_SIZE = 50000

print(f'Positives (train):      {train_pos_cnt:,}')
print(f'Negative pool (train):  {total_neg_pool:,}')
print(f'Target negatives:       {max_train_negs:,}  (ratio {HARD_NEG_RATIO})')
print(f'Hard-neg chunk size:    {HARD_NEG_CHUNK_SIZE:,}')

from rapidfuzz import fuzz as _fuzz

CATEGORIES = [
    'high_name_low_addr',
    'high_addr_low_name',
    'high_name_conflict_door',
    'high_addr_diff_name',
    'generic_name_collision',
    'remaining_difficult',
]

def _assign_category(ns, as_, s1d, td, s1n):
    if ns >= 80 and as_ < 40:                        return 'high_name_low_addr'
    elif as_ >= 75 and ns < 40:                      return 'high_addr_low_name'
    elif ns >= 70 and s1d and td and s1d != td:      return 'high_name_conflict_door'
    elif as_ >= 70 and ns < 60:                      return 'high_addr_diff_name'
    elif len(s1n.replace(' ','')) <= 5:              return 'generic_name_collision'
    else:                                            return 'remaining_difficult'

NEG_COLS = ('s1_id, target_id, pass_count, split, s1_country, s1_raw_name, s1_raw_addr,'
            ' s1_norm_name, s1_norm_addr, s1_postal, s1_door,'
            ' t_country, t_raw_name, t_raw_addr, t_norm_name, t_norm_addr, t_postal, t_door,'
            ' jaccard_score, candidate_rank')
NEG_QUERY = f"SELECT {NEG_COLS} FROM labeled_topk WHERE split='train' AND label=0"
n_neg_chunks = (total_neg_pool + HARD_NEG_CHUNK_SIZE - 1) // HARD_NEG_CHUNK_SIZE

# ── PASS 1: count category frequencies ────────────────────────────────────
print('\nPass 1/3: counting hard-negative categories...')
cat_counts = {c: 0 for c in CATEGORIES}
for ci in range(n_neg_chunks):
    chunk = con.execute(
        f'{NEG_QUERY} LIMIT {HARD_NEG_CHUNK_SIZE} OFFSET {ci * HARD_NEG_CHUNK_SIZE}'
    ).df()
    s1n_v = chunk['s1_norm_name'].fillna('').values
    tn_v  = chunk['t_norm_name'].fillna('').values
    s1a_v = chunk['s1_norm_addr'].fillna('').values
    ta_v  = chunk['t_norm_addr'].fillna('').values
    s1d_v = chunk['s1_door'].fillna('').values
    td_v  = chunk['t_door'].fillna('').values
    for i in range(len(chunk)):
        ns  = _fuzz.token_sort_ratio(s1n_v[i], tn_v[i])
        as_ = _fuzz.token_sort_ratio(s1a_v[i], ta_v[i])
        cat_counts[_assign_category(ns, as_, s1d_v[i], td_v[i], s1n_v[i])] += 1
    del chunk
    gc.collect()

print('\nHard-Negative Category Counts (full pool):')
for cat in CATEGORIES:
    print(f'  {cat:<30}: {cat_counts[cat]:>8,}')

# ── PASS 2: compute per-category sample targets ────────────────────────────
print('\nPass 2/3: computing per-category sample targets...')
cat_targets = {}
remaining = max_train_negs
cats_with_data = [c for c in CATEGORIES if cat_counts[c] > 0]
for idx, cat in enumerate(cats_with_data):
    if idx == len(cats_with_data) - 1:
        share = remaining
    else:
        share = int(max_train_negs * cat_counts[cat] / total_neg_pool)
    share = min(share, cat_counts[cat], remaining)
    cat_targets[cat] = share
    remaining -= share

print('Target samples per category:')
for cat in CATEGORIES:
    print(f'  {cat:<30}: {cat_targets.get(cat, 0):>8,}')

# ── PASS 3: sample and write selected negatives to DuckDB ─────────────────
print('\nPass 3/3: sampling negatives and writing to DuckDB...')
con.execute("""
CREATE OR REPLACE TABLE hard_negatives_selected (
    s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,
    split VARCHAR, s1_country VARCHAR,
    s1_raw_name VARCHAR, s1_raw_addr VARCHAR,
    s1_norm_name VARCHAR, s1_norm_addr VARCHAR,
    s1_postal VARCHAR, s1_door VARCHAR,
    t_country VARCHAR,
    t_raw_name VARCHAR, t_raw_addr VARCHAR,
    t_norm_name VARCHAR, t_norm_addr VARCHAR,
    t_postal VARCHAR, t_door VARCHAR,
    jaccard_score DOUBLE, candidate_rank BIGINT
);
""")

cat_filled   = {c: 0 for c in CATEGORIES}
total_written = 0
rng = __import__('numpy').random.default_rng(SEED)

for ci in range(n_neg_chunks):
    chunk = con.execute(
        f'{NEG_QUERY} LIMIT {HARD_NEG_CHUNK_SIZE} OFFSET {ci * HARD_NEG_CHUNK_SIZE}'
    ).df()
    s1n_v = chunk['s1_norm_name'].fillna('').values
    tn_v  = chunk['t_norm_name'].fillna('').values
    s1a_v = chunk['s1_norm_addr'].fillna('').values
    ta_v  = chunk['t_norm_addr'].fillna('').values
    s1d_v = chunk['s1_door'].fillna('').values
    td_v  = chunk['t_door'].fillna('').values

    chunk_cats = []
    for i in range(len(chunk)):
        ns  = _fuzz.token_sort_ratio(s1n_v[i], tn_v[i])
        as_ = _fuzz.token_sort_ratio(s1a_v[i], ta_v[i])
        chunk_cats.append(_assign_category(ns, as_, s1d_v[i], td_v[i], s1n_v[i]))
    chunk['_cat'] = chunk_cats

    rows_to_insert = []
    for cat in CATEGORIES:
        need = cat_targets.get(cat, 0) - cat_filled[cat]
        if need <= 0:
            continue
        cat_rows = chunk[chunk['_cat'] == cat]
        if len(cat_rows) == 0:
            continue
        # Proportional share of this category's budget from this chunk
        chunk_share = min(need,
                          max(1, int(cat_targets.get(cat, 0) * len(cat_rows) / max(cat_counts[cat], 1))))
        chunk_share = min(chunk_share, len(cat_rows), need)
        if chunk_share > 0:
            sampled = cat_rows.sample(n=chunk_share,
                                      random_state=int(rng.integers(0, 2**31)))
            rows_to_insert.append(sampled.drop(columns=['_cat']))
            cat_filled[cat] += chunk_share

    if rows_to_insert:
        batch = __import__('pandas').concat(rows_to_insert, ignore_index=True)
        con.register('_neg_batch', batch)
        con.execute('INSERT INTO hard_negatives_selected SELECT * FROM _neg_batch')
        total_written += len(batch)
        del batch

    del chunk, rows_to_insert
    gc.collect()

# ── REPORT ────────────────────────────────────────────────────────────────
final_neg_cnt = con.execute('SELECT count(*) FROM hard_negatives_selected').fetchone()[0]
print('\nSampled Hard-Negative Counts by Category:')
for cat in CATEGORIES:
    print(f'  {cat:<30}: {cat_filled[cat]:>8,}')
print(f'  {"TOTAL":<30}: {final_neg_cnt:>8,}')

assert final_neg_cnt <= train_pos_cnt * HARD_NEG_RATIO + 1, \
    f'HARD NEG ERROR: {final_neg_cnt} > {train_pos_cnt} * {HARD_NEG_RATIO}'
print(f'  [PASS] Neg/Pos ratio: {final_neg_cnt/train_pos_cnt:.3f} (target {HARD_NEG_RATIO})')

try:
    import psutil
    print(f'  Process RAM: {psutil.Process().memory_info().rss/1024/1024:.1f}MB')
except ImportError:
    pass

# ── BUILD train_pairs_selected IN DUCKDB (no pandas) ──────────────────────
con.execute("""
CREATE OR REPLACE TABLE train_pairs_selected AS
SELECT s1_id, target_id, pass_count, split, s1_country, s1_raw_name, s1_raw_addr,
       s1_norm_name, s1_norm_addr, s1_postal, s1_door,
       t_country, t_raw_name, t_raw_addr, t_norm_name, t_norm_addr, t_postal, t_door,
       jaccard_score, candidate_rank, 1 as label
FROM labeled_topk WHERE split='train' AND label=1
UNION ALL
SELECT s1_id, target_id, pass_count, split, s1_country, s1_raw_name, s1_raw_addr,
       s1_norm_name, s1_norm_addr, s1_postal, s1_door,
       t_country, t_raw_name, t_raw_addr, t_norm_name, t_norm_addr, t_postal, t_door,
       jaccard_score, candidate_rank, 0 as label
FROM hard_negatives_selected;
""")

final_train_cnt = con.execute('SELECT count(*) FROM train_pairs_selected').fetchone()[0]
print(f'\nBalanced Training Set:')
print(f'  Positive Pairs:  {train_pos_cnt:,}')
print(f'  Hard Negatives:  {final_neg_cnt:,}')
print(f'  Ratio (Neg:Pos): {final_neg_cnt/train_pos_cnt:.2f}:1')
print(f'  Total Pairs:     {final_train_cnt:,}')


### Section 10: Feature Engineering (24 Features) + Chunked Extraction + Feature Validation

**Memory Architecture**: Training and validation pairs are fetched from DuckDB in chunks
of `FEATURE_CHUNK_SIZE` rows. Only one chunk is in pandas RAM at a time.
The final `X_train` / `X_val` are pre-allocated float32 NumPy arrays.

Before allocation, expected memory is printed:
`rows x 24 features x 4 bytes`

After extraction:
- intermediate DataFrames are deleted and `gc.collect()` is called
- only `X_train`, `y_train`, `X_val`, `y_val` remain in memory
- `val_meta` (s1_id, target_id, label) is kept as a minimal array for threshold search

Feature validation checks (on X_train):
- No NaN / inf values
- Feature min / max / mean
- Constant features (zero variance)
- Highly correlated features (|r| > 0.98)
- Feature name alignment


In [ ]:
import gc

FEATURE_NAMES = [
    'name_exact_raw','name_exact_norm','name_token_sort_ratio','name_token_set_ratio',
    'name_jaro_winkler','name_levenshtein_ratio','name_char_sim','name_substring_containment',
    'addr_exact_norm','addr_token_sort_ratio','addr_token_set_ratio','addr_token_jaccard',
    'addr_char_sim','addr_num_overlap','addr_postal_match',
    'country_agreement','s1_name_missing','t_name_missing','s1_addr_missing','t_addr_missing',
    'candidate_rank','blocking_pass_count','name_x_addr','name_diff_addr'
]
N_FEAT = len(FEATURE_NAMES)
FEATURE_CHUNK_SIZE = 50000

from rapidfuzz import fuzz, distance

def compute_pair_features(s1_raw,s1_norm,s1_addr,s1_postal,s1_door,
                           t_raw,t_norm,t_addr,t_postal,t_door,
                           s1_country,t_country,rank,pass_count):
    exact_raw  = 1.0 if s1_raw and s1_raw==t_raw else 0.0
    exact_norm = 1.0 if s1_norm and s1_norm==t_norm else 0.0
    name_sort  = fuzz.token_sort_ratio(s1_norm,t_norm)/100.0
    name_set   = fuzz.token_set_ratio(s1_norm,t_norm)/100.0
    jw         = distance.JaroWinkler.similarity(s1_norm,t_norm)
    lev        = distance.Levenshtein.normalized_similarity(s1_norm,t_norm)
    name_char  = fuzz.ratio(s1_norm,t_norm)/100.0
    name_sub   = 1.0 if (s1_norm and t_norm and (s1_norm in t_norm or t_norm in s1_norm)) else 0.0
    addr_exact = 1.0 if s1_addr and s1_addr==t_addr else 0.0
    addr_sort  = fuzz.token_sort_ratio(s1_addr,t_addr)/100.0
    addr_set   = fuzz.token_set_ratio(s1_addr,t_addr)/100.0
    s1_at = set(s1_addr.split()) if s1_addr else set()
    t_at  = set(t_addr.split())  if t_addr  else set()
    addr_jac   = len(s1_at&t_at)/len(s1_at|t_at) if (s1_at or t_at) else 0.0
    addr_char  = fuzz.ratio(s1_addr,t_addr)/100.0
    s1_nums = {w for w in s1_at if w.isdigit()}
    t_nums  = {w for w in t_at  if w.isdigit()}
    if s1_nums and t_nums:           num_ov = len(s1_nums&t_nums)/len(s1_nums|t_nums)
    elif not s1_nums and not t_nums: num_ov = 0.5
    else:                            num_ov = 0.0
    postal_m   = 1.0 if s1_postal and t_postal and s1_postal==t_postal else 0.0
    country_ag = 1.0 if s1_country==t_country else 0.0
    s1nm = 1.0 if not s1_norm else 0.0
    tnm  = 1.0 if not t_norm  else 0.0
    s1am = 1.0 if not s1_addr else 0.0
    tam  = 1.0 if not t_addr  else 0.0
    rank_f = 1.0/rank if rank>0 else 0.0
    pc_f   = float(pass_count)
    nxa    = name_sort*addr_sort
    nda    = abs(name_sort-addr_sort)
    return [exact_raw,exact_norm,name_sort,name_set,jw,lev,name_char,name_sub,
            addr_exact,addr_sort,addr_set,addr_jac,addr_char,num_ov,postal_m,
            country_ag,s1nm,tnm,s1am,tam,rank_f,pc_f,nxa,nda]

def _fill_X_from_chunk(df_chunk, X, y, row_offset):
    """Fill pre-allocated X/y arrays from a pandas chunk. No copy returned."""
    n = len(df_chunk)
    s1_raw  = df_chunk['s1_raw_name'].fillna('').values
    s1_norm = df_chunk['s1_norm_name'].fillna('').values
    s1_addr = df_chunk['s1_norm_addr'].fillna('').values
    s1_post = df_chunk['s1_postal'].fillna('').values
    s1_door = df_chunk['s1_door'].fillna('').values
    s1_coun = df_chunk['s1_country'].fillna('').values
    t_raw   = df_chunk['t_raw_name'].fillna('').values
    t_norm  = df_chunk['t_norm_name'].fillna('').values
    t_addr  = df_chunk['t_norm_addr'].fillna('').values
    t_post  = df_chunk['t_postal'].fillna('').values
    t_door  = df_chunk['t_door'].fillna('').values
    t_coun  = df_chunk['t_country'].fillna('').values
    ranks   = df_chunk['candidate_rank'].values
    passes  = df_chunk['pass_count'].values
    labels  = df_chunk['label'].values
    for i in range(n):
        X[row_offset+i] = compute_pair_features(
            str(s1_raw[i] or ''), str(s1_norm[i] or ''), str(s1_addr[i] or ''),
            str(s1_post[i] or ''), str(s1_door[i] or ''),
            str(t_raw[i] or ''),  str(t_norm[i] or ''),  str(t_addr[i] or ''),
            str(t_post[i] or ''), str(t_door[i] or ''),
            str(s1_coun[i] or ''), str(t_coun[i] or ''),
            int(ranks[i] or 1), int(passes[i] or 1)
        )
        y[row_offset+i] = int(labels[i])

# ── MEMORY REPORT ─────────────────────────────────────────────────────────
def _mem_report(label, n_rows, n_cols=N_FEAT, dtype_bytes=4):
    mb = n_rows * n_cols * dtype_bytes / 1024 / 1024
    try:
        import psutil
        proc_mb = psutil.Process().memory_info().rss / 1024 / 1024
        print(f'  {label:<35} rows={n_rows:>9,}  cols={n_cols}  est={mb:>7.1f}MB  proc={proc_mb:>7.1f}MB')
    except ImportError:
        print(f'  {label:<35} rows={n_rows:>9,}  cols={n_cols}  est={mb:>7.1f}MB')

print('=' * 70)
print('MEMORY REPORT — before feature allocation')
print('=' * 70)
raw_pairs_cnt  = con.execute('SELECT count(*) FROM raw_candidate_pairs').fetchone()[0]
topk_cnt       = con.execute('SELECT count(*) FROM topk_candidates').fetchone()[0]
train_cnt      = con.execute('SELECT count(*) FROM train_pairs_selected').fetchone()[0]
val_cnt        = con.execute("SELECT count(*) FROM labeled_topk WHERE split='val'").fetchone()[0]
_mem_report('Raw candidate pairs (IDs only)',  raw_pairs_cnt, n_cols=3)
_mem_report('Top-K candidate pairs (full row)', topk_cnt,     n_cols=20)
_mem_report('Training pairs (full row)',         train_cnt,    n_cols=20)
_mem_report('Validation pairs (full row)',        val_cnt,     n_cols=20)
_mem_report('X_train (float32)',                 train_cnt,    n_cols=N_FEAT, dtype_bytes=4)
_mem_report('X_val   (float32)',                 val_cnt,      n_cols=N_FEAT, dtype_bytes=4)
print()

# ── CHUNKED TRAINING FEATURE EXTRACTION ───────────────────────────────────
print(f'Chunked training feature extraction (chunk={FEATURE_CHUNK_SIZE:,})...')
t_feat = time.time()

X_train = np.empty((train_cnt, N_FEAT), dtype=np.float32)
y_train = np.empty(train_cnt, dtype=np.int32)

TRAIN_COLS = ('s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,'
              's1_norm_name,s1_norm_addr,s1_postal,s1_door,'
              't_country,t_raw_name,t_raw_addr,t_norm_name,t_norm_addr,t_postal,t_door,'
              'jaccard_score,candidate_rank,label')

offset = 0
n_train_chunks = (train_cnt + FEATURE_CHUNK_SIZE - 1) // FEATURE_CHUNK_SIZE
for ci in range(n_train_chunks):
    chunk_df = con.execute(
        f'SELECT {TRAIN_COLS} FROM train_pairs_selected'
        f' LIMIT {FEATURE_CHUNK_SIZE} OFFSET {ci * FEATURE_CHUNK_SIZE}'
    ).df()
    _fill_X_from_chunk(chunk_df, X_train, y_train, offset)
    offset += len(chunk_df)
    del chunk_df
    gc.collect()
    if (ci + 1) % 5 == 0 or ci == n_train_chunks - 1:
        print(f'  train chunk {ci+1}/{n_train_chunks}  rows filled: {offset:,}')

# ── CHUNKED VALIDATION FEATURE EXTRACTION ─────────────────────────────────
# Retain only minimal columns needed for threshold search + labels
# Text columns for error analysis are re-fetched from DuckDB later (on demand)
print(f'Chunked validation feature extraction (chunk={FEATURE_CHUNK_SIZE:,})...')

X_val   = np.empty((val_cnt, N_FEAT), dtype=np.float32)
y_val   = np.empty(val_cnt, dtype=np.int32)
# val_meta: minimal columns for threshold search — s1_id, target_id, label
val_s1_ids  = np.empty(val_cnt, dtype=object)
val_tgt_ids = np.empty(val_cnt, dtype=object)

VAL_COLS = ('s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,'
            's1_norm_name,s1_norm_addr,s1_postal,s1_door,'
            't_country,t_raw_name,t_raw_addr,t_norm_name,t_norm_addr,t_postal,t_door,'
            'jaccard_score,candidate_rank,label')

offset = 0
n_val_chunks = (val_cnt + FEATURE_CHUNK_SIZE - 1) // FEATURE_CHUNK_SIZE
for ci in range(n_val_chunks):
    chunk_df = con.execute(
        f"SELECT {VAL_COLS} FROM labeled_topk WHERE split='val'"
        f' LIMIT {FEATURE_CHUNK_SIZE} OFFSET {ci * FEATURE_CHUNK_SIZE}'
    ).df()
    _fill_X_from_chunk(chunk_df, X_val, y_val, offset)
    val_s1_ids [offset:offset+len(chunk_df)] = chunk_df['s1_id'].values
    val_tgt_ids[offset:offset+len(chunk_df)] = chunk_df['target_id'].values
    offset += len(chunk_df)
    del chunk_df
    gc.collect()
    if (ci + 1) % 5 == 0 or ci == n_val_chunks - 1:
        print(f'  val chunk {ci+1}/{n_val_chunks}  rows filled: {offset:,}')

print(f'Feature extraction complete in {time.time()-t_feat:.1f}s')
print(f'  X_train: {X_train.shape}  X_val: {X_val.shape}')
print(f'  X_train RAM: {X_train.nbytes/1024/1024:.1f}MB  X_val RAM: {X_val.nbytes/1024/1024:.1f}MB')

# ── FEATURE VALIDATION ────────────────────────────────────────────────────
print('\n' + '='*60)
print('FEATURE VALIDATION')
print('='*60)

nan_count = int(np.isnan(X_train).sum())
inf_count = int(np.isinf(X_train).sum())
print(f'NaN values in X_train: {nan_count}')
print(f'Inf values in X_train: {inf_count}')
if nan_count > 0 or inf_count > 0:
    raise ValueError('FEATURE VALIDATION FAILED: NaN or Inf detected in X_train!')
print('  [PASS] No NaN or Inf values.')

print('\nFeature statistics (X_train):')
print(f'  {"Feature":<30} {"Min":>8} {"Max":>8} {"Mean":>8}')
print(f'  {"-"*30} {"-"*8} {"-"*8} {"-"*8}')
for i, fname in enumerate(FEATURE_NAMES):
    print(f'  {fname:<30} {X_train[:,i].min():>8.4f} {X_train[:,i].max():>8.4f} {X_train[:,i].mean():>8.4f}')

constant_feats = [FEATURE_NAMES[i] for i in range(N_FEAT) if X_train[:,i].std() == 0]
print(f'\nConstant features: {constant_feats if constant_feats else "None"}')
if constant_feats:
    print(f'  [WARN] {len(constant_feats)} constant feature(s).')
else:
    print('  [PASS] No constant features.')

corr_matrix = np.corrcoef(X_train.T)
redundant_pairs = []
for i in range(N_FEAT):
    for j in range(i+1, N_FEAT):
        if abs(corr_matrix[i,j]) > 0.98:
            redundant_pairs.append((FEATURE_NAMES[i], FEATURE_NAMES[j], round(float(corr_matrix[i,j]),4)))
if redundant_pairs:
    print(f'\nHighly correlated pairs (|r|>0.98):')
    for a,b,r in redundant_pairs:
        print(f'  {a} <-> {b}  r={r}')
    print(f'  [WARN] {len(redundant_pairs)} pair(s) — informational only.')
else:
    print('\n  [PASS] No highly correlated feature pairs (|r|>0.98).')

if N_FEAT != X_train.shape[1]:
    raise ValueError(f'FEATURE VALIDATION FAILED: {N_FEAT} names but {X_train.shape[1]} columns!')
print(f'\n  [PASS] Feature name alignment: {N_FEAT} names == {X_train.shape[1]} columns.')
print('\nFeature validation complete. Proceeding to model training.')


### Section 11 & 12: Model Training — Logistic Regression vs. XGBoost

Both models use identical training pairs, validation pairs, and features.
No large hyperparameter search is performed.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

print('Training Model A: Logistic Regression...')
t_lr = time.time()
scaler = StandardScaler()
# fit_transform creates a float64 copy of X_train internally.
# Additional RAM = X_train rows x N_FEAT x 8 bytes (float64)
_lr_extra_mb = X_train.shape[0] * X_train.shape[1] * 8 / 1024 / 1024
print(f'  LR scaler: fit_transform will use ~{_lr_extra_mb:.1f}MB extra (float64 copy)')
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
lr_model = LogisticRegression(C=1.0, max_iter=500, random_state=SEED, solver='lbfgs')
lr_model.fit(X_train_sc, y_train)
lr_time = time.time() - t_lr
lr_val_probs = lr_model.predict_proba(X_val_sc)[:,1]
print(f'  Logistic Regression trained in {lr_time:.2f}s')

print('Training Model B: XGBoost...')
t_xgb = time.time()
xgb_model = xgb.XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.08,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', random_state=SEED, n_jobs=4
)
xgb_model.fit(X_train, y_train)
xgb_time = time.time() - t_xgb
xgb_val_probs = xgb_model.predict_proba(X_val)[:,1]
print(f'  XGBoost trained in {xgb_time:.2f}s')

importances = xgb_model.feature_importances_
top_idx = np.argsort(importances)[::-1][:10]
print('\nTop 10 XGBoost Feature Importances:')
for r, idx in enumerate(top_idx, 1):
    print(f'  {r:>2}. {FEATURE_NAMES[idx]:<28}: {importances[idx]*100:5.2f}%')


### Section 13 & 14: Macro F0.5 — Unit Tests + Threshold Search

**Unit tests** verify the F0.5 implementation against 6 manually constructed cases before
running the threshold grid. The threshold grid uses validation data only.

Cases tested:
- A: true empty + predicted empty → F0.5 = 1.0
- B: true empty + false prediction → F0.5 = 0.0
- C: one true match + correct prediction → F0.5 = 1.0
- D: one true match + wrong prediction → F0.5 = 0.0
- E: multiple true matches + partial prediction → 0 < F0.5 < 1
- F: multiple true matches + extra false positive → F0.5 penalised


In [ ]:
from collections import defaultdict
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

def compute_entity_f05(true_matches, pred_matches):
    """Compute per-entity F0.5. true_matches and pred_matches are sets."""
    if not true_matches:
        return 1.0 if not pred_matches else 0.0
    if not pred_matches:
        return 0.0
    tp = len(true_matches & pred_matches)
    if tp == 0:
        return 0.0
    prec = tp / len(pred_matches)
    rec  = tp / len(true_matches)
    denom = 0.25 * prec + rec
    return (1.25 * prec * rec) / denom if denom > 0 else 0.0

# ── UNIT TESTS ────────────────────────────────────────────────────────────
print('=' * 60)
print('F0.5 UNIT TESTS')
print('=' * 60)

unit_tests = [
    ('A: true empty + pred empty',        set(),       set(),       1.0),
    ('B: true empty + false pred',         set(),       {'x1'},      0.0),
    ('C: one true match + correct pred',   {'m1'},      {'m1'},      1.0),
    ('D: one true match + wrong pred',     {'m1'},      {'m2'},      0.0),
    ('E: multi true + partial pred',       {'m1','m2'}, {'m1'},      None),
    ('F: multi true + extra FP',           {'m1','m2'}, {'m1','m2','fp1'}, None),
]

all_passed = True
for name, true_s, pred_s, expected in unit_tests:
    got = compute_entity_f05(true_s, pred_s)
    if expected is not None:
        ok = abs(got - expected) < 1e-9
        status = '[PASS]' if ok else '[FAIL]'
        if not ok: all_passed = False
        print(f'  {status} {name}: expected={expected:.4f} got={got:.4f}')
    else:
        # Sanity: must be in (0,1)
        ok = 0.0 < got < 1.0
        status = '[PASS]' if ok else '[FAIL]'
        if not ok: all_passed = False
        print(f'  {status} {name}: got={got:.4f} (expected 0 < F0.5 < 1)')

# Case E explicit check: partial recall should give specific value
e_got = compute_entity_f05({'m1','m2'}, {'m1'})
# tp=1, prec=1.0, rec=0.5 => F0.5 = 1.25*1.0*0.5/(0.25*1.0+0.5) = 0.625/0.75 = 0.8333
e_exp = 1.25*1.0*0.5/(0.25*1.0+0.5)
ok_e = abs(e_got - e_exp) < 1e-6
print(f'  {"[PASS]" if ok_e else "[FAIL]"} Case E exact value: expected={e_exp:.4f} got={e_got:.4f}')
if not ok_e: all_passed = False

# Case F: tp=2, prec=2/3, rec=1.0 => F0.5 = 1.25*(2/3)*1.0/(0.25*(2/3)+1.0)
f_got = compute_entity_f05({'m1','m2'}, {'m1','m2','fp1'})
f_exp = 1.25*(2/3)*1.0/(0.25*(2/3)+1.0)
ok_f = abs(f_got - f_exp) < 1e-6
print(f'  {"[PASS]" if ok_f else "[FAIL]"} Case F exact value: expected={f_exp:.4f} got={f_got:.4f}')
if not ok_f: all_passed = False

if not all_passed:
    raise AssertionError('F0.5 unit tests FAILED. Fix compute_entity_f05 before proceeding.')
print('All F0.5 unit tests passed.\n')

# ── BUILD VALIDATION GT MAP ────────────────────────────────────────────────
# Every validation S1 must be evaluated (including singletons)
val_gt_rows = con.execute("SELECT entity_id, matched_entity_ids FROM s1_split WHERE split='val'").fetchall()
val_gt = {}
for sid, mstr in val_gt_rows:
    if mstr and str(mstr).strip():
        val_gt[sid] = {m.strip() for m in str(mstr).split(',') if m.strip()}
    else:
        val_gt[sid] = set()  # singleton: true empty set
print(f'Validation S1 entities in GT map: {len(val_gt):,}')
print(f'  Singletons (true empty): {sum(1 for v in val_gt.values() if not v):,}')
print(f'  Non-singletons:          {sum(1 for v in val_gt.values() if v):,}')

# Verify all val S1s have candidates (or will get empty prediction)
val_s1_with_cands = set(r[0] for r in con.execute("SELECT DISTINCT s1_id FROM labeled_topk WHERE split='val'").fetchall())
val_s1_no_cands = set(val_gt.keys()) - val_s1_with_cands
print(f'  Val S1s with no candidates (will predict empty): {len(val_s1_no_cands):,}')

# ── THRESHOLD GRID SEARCH ─────────────────────────────────────────────────
def evaluate_threshold_grid(val_probs, model_name):
    scores_by_s1 = defaultdict(list)
    for s1_id, tgt_id, prob in zip(val_s1_ids, val_tgt_ids, val_probs):
        scores_by_s1[s1_id].append((tgt_id, float(prob)))

    thresholds = [round(t,2) for t in np.arange(0.30, 0.96, 0.05)]
    results, best_row, best_f05 = [], None, -1.0

    for tau in thresholds:
        f05_list = []
        total_tp = total_fp = total_fn = total_preds = 0
        # Evaluate EVERY validation S1 (including singletons and those with no candidates)
        for sid, true_set in val_gt.items():
            cands    = scores_by_s1.get(sid, [])
            pred_set = {tid for tid, prob in cands if prob >= tau}
            f05_list.append(compute_entity_f05(true_set, pred_set))
            total_tp    += len(true_set & pred_set)
            total_fp    += len(pred_set - true_set)
            total_fn    += len(true_set - pred_set)
            total_preds += len(pred_set)

        macro_f05 = float(np.mean(f05_list))
        prec = total_tp/(total_tp+total_fp) if (total_tp+total_fp)>0 else 0.0
        rec  = total_tp/(total_tp+total_fn) if (total_tp+total_fn)>0 else 0.0
        row  = {'model': model_name, 'threshold': tau,
                'macro_f05': round(macro_f05,4),
                'precision': round(prec,4), 'recall': round(rec,4),
                'false_positives': total_fp, 'false_negatives': total_fn,
                'avg_matches_per_s1': round(total_preds/len(val_gt),3),
                'n_val_s1_evaluated': len(val_gt)}
        results.append(row)
        if macro_f05 > best_f05:
            best_f05, best_row = macro_f05, row
    return pd.DataFrame(results), best_row

lr_df,  lr_best  = evaluate_threshold_grid(lr_val_probs,  'LogisticRegression')
xgb_df, xgb_best = evaluate_threshold_grid(xgb_val_probs, 'XGBoost')

lr_roc  = roc_auc_score(y_val, lr_val_probs)
lr_pr   = average_precision_score(y_val, lr_val_probs)
xgb_roc = roc_auc_score(y_val, xgb_val_probs)
xgb_pr  = average_precision_score(y_val, xgb_val_probs)
lr_f1   = f1_score(y_val, (lr_val_probs  >= lr_best['threshold']).astype(int))
xgb_f1  = f1_score(y_val, (xgb_val_probs >= xgb_best['threshold']).astype(int))

comp_summary = pd.DataFrame([
    {'Model':'Logistic Regression','Optimal tau*':lr_best['threshold'],
     'Macro F0.5':lr_best['macro_f05'],'Precision':lr_best['precision'],
     'Recall':lr_best['recall'],'F1':round(lr_f1,4),
     'ROC-AUC':round(lr_roc,4),'PR-AUC':round(lr_pr,4),
     'Val S1 Evaluated':lr_best['n_val_s1_evaluated'],
     'Train Time':f'{lr_time:.2f}s'},
    {'Model':'XGBoost','Optimal tau*':xgb_best['threshold'],
     'Macro F0.5':xgb_best['macro_f05'],'Precision':xgb_best['precision'],
     'Recall':xgb_best['recall'],'F1':round(xgb_f1,4),
     'ROC-AUC':round(xgb_roc,4),'PR-AUC':round(xgb_pr,4),
     'Val S1 Evaluated':xgb_best['n_val_s1_evaluated'],
     'Train Time':f'{xgb_time:.2f}s'},
])
print('=' * 80)
print('FINAL VALIDATION MODEL COMPARISON (Exact Macro F0.5, threshold on val only)')
print('=' * 80)
display(comp_summary)


### Section 15: Error Analysis


In [ ]:
winning_model = 'LogisticRegression' if lr_best['macro_f05'] >= xgb_best['macro_f05'] else 'XGBoost'
winning_best  = lr_best  if winning_model == 'LogisticRegression' else xgb_best
winning_probs = lr_val_probs if winning_model == 'LogisticRegression' else xgb_val_probs
print(f'Winning model: {winning_model} at tau*={winning_best["threshold"]:.2f}')

# Determine predicted match using minimal val arrays (no full DataFrame needed)
tau_star = winning_best['threshold']
pred_match = (winning_probs >= tau_star).astype(np.int32)

# Identify FP/FN indices in the val arrays
fp_mask = (y_val == 0) & (pred_match == 1)
fn_mask = (y_val == 1) & (pred_match == 0)
print(f'False Positives: {fp_mask.sum():,}  False Negatives: {fn_mask.sum():,}')

# Re-fetch only the text columns needed for FP analysis from DuckDB (on demand)
# This avoids keeping the full val DataFrame in memory throughout training
fp_s1_ids  = val_s1_ids[fp_mask].tolist()
fp_tgt_ids = val_tgt_ids[fp_mask].tolist()

if fp_s1_ids:
    import pandas as _pd
    con.register('fp_s1_ids_df',  _pd.DataFrame({'s1_id':  fp_s1_ids}))
    con.register('fp_tgt_ids_df', _pd.DataFrame({'target_id': fp_tgt_ids}))
    df_fp_text = con.execute("""
        SELECT tk.s1_id, tk.target_id,
               tk.s1_norm_name, tk.t_norm_name,
               tk.s1_norm_addr, tk.t_norm_addr,
               tk.s1_door, tk.t_door
        FROM labeled_topk tk
        WHERE tk.s1_id    IN (SELECT s1_id    FROM fp_s1_ids_df)
          AND tk.target_id IN (SELECT target_id FROM fp_tgt_ids_df)
          AND tk.split = 'val' AND tk.label = 0
    """).df()
else:
    df_fp_text = __import__('pandas').DataFrame()

fp_cats = defaultdict(int)
for _, r in df_fp_text.iterrows():
    ns  = fuzz.token_sort_ratio(str(r.get('s1_norm_name','') or ''), str(r.get('t_norm_name','') or ''))
    as_ = fuzz.token_sort_ratio(str(r.get('s1_norm_addr','') or ''), str(r.get('t_norm_addr','') or ''))
    sd  = str(r.get('s1_door','') or '')
    td  = str(r.get('t_door','') or '')
    sn  = str(r.get('s1_norm_name','') or '')
    if ns >= 85 and as_ < 40:                 fp_cats['same_name_diff_address'] += 1
    elif as_ >= 80 and ns < 40:               fp_cats['same_address_diff_business'] += 1
    elif sd and td and sd != td and ns >= 70: fp_cats['address_number_conflict'] += 1
    elif len(sn.replace(' ','')) <= 5:        fp_cats['abbreviation_collision'] += 1
    else:                                     fp_cats['generic_name_collision'] += 1

fp_total = max(len(df_fp_text), 1)
fp_summary = pd.DataFrame([{'Category':k,'Count':v,'Pct':f'{v/fp_total*100:.1f}%'}
                            for k,v in sorted(fp_cats.items(), key=lambda x:-x[1])])
print('\nFalse Positive Categories:')
display(fp_summary)

# Clean up FP text frame — not needed after this cell
del df_fp_text
gc.collect()


### Section 16 & 17: Save Model Artifacts and Results


In [ ]:
import joblib

artifacts_dir = 'artifacts'
results_dir   = 'results'
os.makedirs(os.path.join(artifacts_dir,'logistic_regression'), exist_ok=True)
os.makedirs(os.path.join(artifacts_dir,'xgboost'), exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

joblib.dump(lr_model, os.path.join(artifacts_dir,'logistic_regression','model.joblib'))
joblib.dump(scaler,   os.path.join(artifacts_dir,'logistic_regression','scaler.joblib'))
xgb_model.save_model(os.path.join(artifacts_dir,'xgboost','xgboost_model.json'))

with open(os.path.join(artifacts_dir,'feature_config.json'),'w') as f:
    json.dump({'feature_names':FEATURE_NAMES,'count':len(FEATURE_NAMES)}, f, indent=2)

with open(os.path.join(artifacts_dir,'selected_model.json'),'w') as f:
    json.dump({'selected_model':winning_model,
               'optimal_threshold':winning_best['threshold'],
               'validation_macro_f05':winning_best['macro_f05'],
               'validation_precision':winning_best['precision'],
               'validation_recall':winning_best['recall']}, f, indent=2)

comp_summary.to_csv(os.path.join(results_dir,'model_comparison.csv'), index=False)
pd.concat([lr_df,xgb_df]).to_csv(os.path.join(results_dir,'threshold_results.csv'), index=False)
# Build minimal val predictions frame from arrays (no full DataFrame needed)
import pandas as _pd2
_val_preds = _pd2.DataFrame({
    's1_id':      val_s1_ids[:10000],
    'target_id':  val_tgt_ids[:10000],
    'label':      y_val[:10000],
    'pred_prob':  winning_probs[:10000],
    'pred_match': pred_match[:10000],
})
_val_preds.to_csv(os.path.join(results_dir,'validation_predictions.csv'), index=False)
del _val_preds

print(f'Artifacts saved to {artifacts_dir}/  Results saved to {results_dir}/')


### Section 18: Final Experiment Summary

**This was a DEVELOPMENT EXPERIMENT** (150k S1 entities, random distractors).

1. **Blocking**: Finalized 8-pass architecture, K=25, zero OOM overhead.
2. **Hard Negatives**: 6-category deliberate sampling at HARD_NEG_RATIO=3.0.
3. **Target Pool**: Deduplicated (one unique record per entity_id). Duplicate count before/after reported. True-target %, S2/S3 distractor counts reported.
4. **Leakage**: Explicit checks — no S1 in both splits, no cross-split candidate pairs.
5. **F0.5**: Unit-tested implementation, all val S1s evaluated including singletons.
6. **Feature Validation**: NaN/inf, min/max/mean, constant, redundant features checked.
7. **Models**: LR vs XGBoost on identical splits. Threshold selected on val only.
8. **Reproducibility**: SEED=42, TRAIN_S1_LIMIT=150000, VALIDATION_FRACTION=0.20, TOP_K=25.

---

```
EVALUATION TYPE:
  Conditional matcher validation — not end-to-end test evaluation

  The F0.5 scores above measure how well the matcher ranks and selects
  candidates given that the true target was already present in the pool.
  They do NOT reflect end-to-end competition performance, which also
  depends on blocking recall over the full S2+S3 corpus.
  End-to-end blocking recall is evaluated separately by blocking_experiment.py.
```


### Section 19: Label Baseline & Save Baseline Artifacts

**BASELINE_8PASS_XGBOOST** — 8-pass blocking, K=25, XGBoost.
Known result: Macro F0.5 = 0.8841, Precision = 0.9816, Recall = 0.7777, threshold = 0.80.
Baseline artifacts are written to `/content/artifacts/baseline/` and must not be overwritten.


In [ ]:
import joblib, os, json as _json

BASELINE_TAG = 'BASELINE_8PASS_XGBOOST'
baseline_dir = '/content/artifacts/baseline'
os.makedirs(baseline_dir, exist_ok=True)

# Copy existing XGBoost model as baseline
xgb_model.save_model(os.path.join(baseline_dir, 'xgboost_baseline.json'))
joblib.dump(lr_model, os.path.join(baseline_dir, 'lr_baseline.joblib'))
joblib.dump(scaler,   os.path.join(baseline_dir, 'lr_scaler_baseline.joblib'))

baseline_meta = {
    'tag': BASELINE_TAG,
    'blocking': '8-pass',
    'K': 25,
    'model': 'XGBoost',
    'macro_f05': xgb_best['macro_f05'],
    'precision': xgb_best['precision'],
    'recall':    xgb_best['recall'],
    'threshold': xgb_best['threshold'],
    'n_val_s1':  xgb_best['n_val_s1_evaluated'],
    'known_result': {'macro_f05': 0.8841, 'precision': 0.9816, 'recall': 0.7777, 'threshold': 0.80},
}
with open(os.path.join(baseline_dir, 'baseline_meta.json'), 'w') as f:
    _json.dump(baseline_meta, f, indent=2)

# Also copy existing results CSVs
import shutil
for fn in ['model_comparison.csv', 'threshold_results.csv', 'validation_predictions.csv']:
    src = os.path.join('results', fn)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(baseline_dir, fn))

print(f'Baseline [{BASELINE_TAG}] artifacts saved to {baseline_dir}/')
print(f'  XGBoost Macro F0.5 (measured): {xgb_best["macro_f05"]}')
print(f'  Known result:                   0.8841')
print('Baseline is LOCKED. Subsequent sections write to /content/experiment_v2/ only.')


### Section 20: Install CatBoost


In [ ]:
try:
    import catboost as _cb
    print(f'CatBoost already installed: {_cb.__version__}')
except ImportError:
    print('Installing CatBoost...')
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost>=1.2'])
    import catboost as _cb
    print(f'CatBoost installed: {_cb.__version__}')

from catboost import CatBoostClassifier
print('CatBoost ready.')


### Section 21: Blocking Strategy Comparison

Evaluate multiple blocking strategies on the **same validation S1 entities** used by the baseline.
For each strategy measure: candidate pairs, avg/p95 candidates per S1, recall@K, runtime, peak memory.

Strategies tested:
- **A** — Baseline 8-pass K=25 (already built as `topk_candidates`)
- **B** — 8-pass K=50
- **C** — 8-pass K=100
- **D** — 8-pass + 4 permutation passes, K=50
- **E** — 8-pass + char 3/4-gram TF-IDF sparse retrieval, K=50
- **F** — 8-pass + address-driven passes (postal+door, street tokens), K=50

**Memory safety**: all blocking is done in DuckDB (disk-backed). Char n-gram retrieval uses sparse scipy matrices processed country-by-country. No full Cartesian product is ever materialized.


In [ ]:
import gc, time
import numpy as np
import pandas as pd

# ── helpers ───────────────────────────────────────────────────────────────
def _recall_at_k(con, topk_table, k, val_gt):
    """Candidate recall@k for val S1s that have at least one true match."""
    rows = con.execute(f"""
        SELECT c.s1_id, c.target_id
        FROM {topk_table} c
        WHERE c.split='val' AND c.candidate_rank <= {k}
    """).fetchall()
    cands = {}
    for s1, tgt in rows:
        cands.setdefault(s1, set()).add(tgt)
    hits = 0; total = 0
    for sid, true_set in val_gt.items():
        if not true_set: continue
        total += len(true_set)
        hits  += len(true_set & cands.get(sid, set()))
    return hits / total if total > 0 else 0.0

def _cand_stats(con, topk_table, split='val'):
    rows = con.execute(f"""
        SELECT count(*) as n, count(distinct s1_id) as s1s
        FROM {topk_table} WHERE split='{split}'
    """).fetchone()
    total, s1s = rows
    per = con.execute(f"""
        SELECT count(*) as c FROM {topk_table}
        WHERE split='{split}' GROUP BY s1_id ORDER BY c
    """).fetchall()
    per = [r[0] for r in per]
    avg = sum(per)/len(per) if per else 0
    p95 = per[int(len(per)*0.95)] if per else 0
    return total, s1s, avg, p95

def _mem_mb():
    try:
        import psutil
        return psutil.Process().memory_info().rss / 1024 / 1024
    except ImportError:
        return 0.0

blocking_results = []

# ── Strategy A: baseline 8-pass K=25 (already built) ─────────────────────
print('Strategy A: baseline 8-pass K=25 (pre-built)...')
t0 = time.time()
tot, s1s, avg, p95 = _cand_stats(con, 'topk_candidates')
r25 = _recall_at_k(con, 'topk_candidates', 25, val_gt)
r50 = _recall_at_k(con, 'topk_candidates', 50, val_gt)  # same table, K capped at 25
blocking_results.append({
    'strategy':'A_8pass_K25','K':25,
    'candidate_pairs':tot,'avg_cands':round(avg,1),'p95_cands':p95,
    'recall_at_25':round(r25,4),'recall_at_50':round(r25,4),'recall_at_100':round(r25,4),
    'runtime_s':round(time.time()-t0,1),'peak_mem_mb':round(_mem_mb(),1)
})
print(f'  pairs={tot:,}  avg={avg:.1f}  p95={p95}  recall@25={r25:.4f}')


In [ ]:
# ── Strategy B: 8-pass K=50 ──────────────────────────────────────────────
print('Strategy B: 8-pass K=50...')
t0 = time.time(); m0 = _mem_mb()
K_B = 50
con.execute('CREATE OR REPLACE TABLE topk_B (s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,'
            ' split VARCHAR, s1_country VARCHAR,'
            ' s1_raw_name VARCHAR, s1_raw_addr VARCHAR,'
            ' s1_norm_name VARCHAR, s1_norm_addr VARCHAR,'
            ' s1_postal VARCHAR, s1_door VARCHAR,'
            ' t_country VARCHAR, t_raw_name VARCHAR, t_raw_addr VARCHAR,'
            ' t_norm_name VARCHAR, t_norm_addr VARCHAR,'
            ' t_postal VARCHAR, t_door VARCHAR,'
            ' jaccard_score DOUBLE, candidate_rank BIGINT);')
RANK_CHUNK_SIZE = 10000
s1_ids_all_b = [r[0] for r in con.execute('SELECT DISTINCT s1_id FROM raw_candidate_pairs ORDER BY s1_id').fetchall()]
n_chunks_b = (len(s1_ids_all_b) + RANK_CHUNK_SIZE - 1) // RANK_CHUNK_SIZE
for ci in range(n_chunks_b):
    chunk = s1_ids_all_b[ci*RANK_CHUNK_SIZE:(ci+1)*RANK_CHUNK_SIZE]
    con.register('_cids', pd.DataFrame({'s1_id': chunk}))
    con.execute(f"""
    INSERT INTO topk_B
    SELECT s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,
           s1_norm_name,s1_norm_addr,s1_postal,s1_door,t_country,t_raw_name,t_raw_addr,
           t_norm_name,t_norm_addr,t_postal,t_door,jaccard_score,candidate_rank
    FROM (
        SELECT c.s1_id,c.target_id,c.pass_count,s.split,s.country AS s1_country,
               s.raw_name AS s1_raw_name,s.raw_addr AS s1_raw_addr,
               s.norm_name AS s1_norm_name,s.norm_addr AS s1_norm_addr,
               s.postal AS s1_postal,s.door AS s1_door,
               t.country AS t_country,t.raw_name AS t_raw_name,t.raw_addr AS t_raw_addr,
               t.norm_name AS t_norm_name,t.norm_addr AS t_norm_addr,
               t.postal AS t_postal,t.door AS t_door,
               (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0)))
               AS jaccard_score,
               row_number() OVER (PARTITION BY c.s1_id ORDER BY
                   (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                    0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0))) DESC,
                   c.pass_count DESC) AS candidate_rank
        FROM raw_candidate_pairs c
        JOIN _cids x ON c.s1_id=x.s1_id
        JOIN s1_split s ON c.s1_id=s.entity_id
        JOIN target_pool t ON c.target_id=t.entity_id
    ) ranked WHERE candidate_rank <= {K_B};
    """)
tot,s1s,avg,p95 = _cand_stats(con,'topk_B')
r25 = _recall_at_k(con,'topk_B',25,val_gt)
r50 = _recall_at_k(con,'topk_B',50,val_gt)
r100= _recall_at_k(con,'topk_B',50,val_gt)  # capped at K_B
blocking_results.append({
    'strategy':'B_8pass_K50','K':50,
    'candidate_pairs':tot,'avg_cands':round(avg,1),'p95_cands':p95,
    'recall_at_25':round(r25,4),'recall_at_50':round(r50,4),'recall_at_100':round(r100,4),
    'runtime_s':round(time.time()-t0,1),'peak_mem_mb':round(_mem_mb()-m0,1)
})
print(f'  pairs={tot:,}  avg={avg:.1f}  p95={p95}  recall@25={r25:.4f}  recall@50={r50:.4f}')
gc.collect()

# ── Strategy C: 8-pass K=100 ─────────────────────────────────────────────
print('Strategy C: 8-pass K=100...')
t0 = time.time(); m0 = _mem_mb()
K_C = 100
con.execute('CREATE OR REPLACE TABLE topk_C (s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,'
            ' split VARCHAR, s1_country VARCHAR,'
            ' s1_raw_name VARCHAR, s1_raw_addr VARCHAR,'
            ' s1_norm_name VARCHAR, s1_norm_addr VARCHAR,'
            ' s1_postal VARCHAR, s1_door VARCHAR,'
            ' t_country VARCHAR, t_raw_name VARCHAR, t_raw_addr VARCHAR,'
            ' t_norm_name VARCHAR, t_norm_addr VARCHAR,'
            ' t_postal VARCHAR, t_door VARCHAR,'
            ' jaccard_score DOUBLE, candidate_rank BIGINT);')
for ci in range(n_chunks_b):
    chunk = s1_ids_all_b[ci*RANK_CHUNK_SIZE:(ci+1)*RANK_CHUNK_SIZE]
    con.register('_cids', pd.DataFrame({'s1_id': chunk}))
    con.execute(f"""
    INSERT INTO topk_C
    SELECT s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,
           s1_norm_name,s1_norm_addr,s1_postal,s1_door,t_country,t_raw_name,t_raw_addr,
           t_norm_name,t_norm_addr,t_postal,t_door,jaccard_score,candidate_rank
    FROM (
        SELECT c.s1_id,c.target_id,c.pass_count,s.split,s.country AS s1_country,
               s.raw_name AS s1_raw_name,s.raw_addr AS s1_raw_addr,
               s.norm_name AS s1_norm_name,s.norm_addr AS s1_norm_addr,
               s.postal AS s1_postal,s.door AS s1_door,
               t.country AS t_country,t.raw_name AS t_raw_name,t.raw_addr AS t_raw_addr,
               t.norm_name AS t_norm_name,t.norm_addr AS t_norm_addr,
               t.postal AS t_postal,t.door AS t_door,
               (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0)))
               AS jaccard_score,
               row_number() OVER (PARTITION BY c.s1_id ORDER BY
                   (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                    0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0))) DESC,
                   c.pass_count DESC) AS candidate_rank
        FROM raw_candidate_pairs c
        JOIN _cids x ON c.s1_id=x.s1_id
        JOIN s1_split s ON c.s1_id=s.entity_id
        JOIN target_pool t ON c.target_id=t.entity_id
    ) ranked WHERE candidate_rank <= {K_C};
    """)
tot,s1s,avg,p95 = _cand_stats(con,'topk_C')
r25 = _recall_at_k(con,'topk_C',25,val_gt)
r50 = _recall_at_k(con,'topk_C',50,val_gt)
r100= _recall_at_k(con,'topk_C',100,val_gt)
blocking_results.append({
    'strategy':'C_8pass_K100','K':100,
    'candidate_pairs':tot,'avg_cands':round(avg,1),'p95_cands':p95,
    'recall_at_25':round(r25,4),'recall_at_50':round(r50,4),'recall_at_100':round(r100,4),
    'runtime_s':round(time.time()-t0,1),'peak_mem_mb':round(_mem_mb()-m0,1)
})
print(f'  pairs={tot:,}  avg={avg:.1f}  p95={p95}  recall@25={r25:.4f}  recall@50={r50:.4f}  recall@100={r100:.4f}')
gc.collect()


In [ ]:
# ── Strategy D: 8-pass + 4 permutation passes, K=50 ─────────────────────
# Permutation passes: p2+p1, p1+p3, cmp4 prefix, p2+postal
print('Strategy D: 8-pass + permutation passes, K=50...')
t0 = time.time(); m0 = _mem_mb()
K_D = 50
con.execute("""
CREATE OR REPLACE TABLE raw_cands_D AS
SELECT sub.entity_id as s1_id, sub.target_id, count(*) as pass_count
FROM (
    -- original 8 passes
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 AND s.p2=t.p2 WHERE length(s.p1)>=3 AND length(s.p2)>=3
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 AND s.postal=t.postal WHERE length(s.p1)>=3 AND s.postal!=''
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 AND s.door=t.door WHERE length(s.p1)>=3 AND s.door!=''
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.cmp5=t.cmp5 WHERE length(s.cmp5)>=5
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 WHERE length(s.p1)>=5
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.postal=t.postal AND s.door=t.door WHERE s.postal!='' AND length(s.door)>=2
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.door=t.door AND s.a1=t.a1 WHERE length(s.door)>=2 AND length(s.a1)>=4
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.door=t.door AND s.a2=t.a2 WHERE length(s.door)>=2 AND length(s.a2)>=4
    -- permutation pass 1: p2+p1 (word order swap)
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p2=t.p1 AND s.p1=t.p2 WHERE length(s.p1)>=3 AND length(s.p2)>=3
    -- permutation pass 2: p1+p3
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p1=t.p1 AND s.p3=t.p3 WHERE length(s.p1)>=3 AND length(s.p3)>=3
    -- permutation pass 3: cmp4 prefix
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.cmp4=t.cmp4 WHERE length(s.cmp4)>=4
    -- permutation pass 4: p2+postal
    UNION ALL
    SELECT s.entity_id,t.entity_id as target_id FROM s1_split s JOIN target_pool t ON s.country=t.country AND s.p2=t.p2 AND s.postal=t.postal WHERE length(s.p2)>=3 AND s.postal!=''
) sub
GROUP BY sub.entity_id, sub.target_id;
""")
raw_d = con.execute('SELECT count(*) FROM raw_cands_D').fetchone()[0]
print(f'  Raw pairs D: {raw_d:,}')

con.execute('CREATE OR REPLACE TABLE topk_D (s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,'
            ' split VARCHAR, s1_country VARCHAR,'
            ' s1_raw_name VARCHAR, s1_raw_addr VARCHAR,'
            ' s1_norm_name VARCHAR, s1_norm_addr VARCHAR,'
            ' s1_postal VARCHAR, s1_door VARCHAR,'
            ' t_country VARCHAR, t_raw_name VARCHAR, t_raw_addr VARCHAR,'
            ' t_norm_name VARCHAR, t_norm_addr VARCHAR,'
            ' t_postal VARCHAR, t_door VARCHAR,'
            ' jaccard_score DOUBLE, candidate_rank BIGINT);')
s1_ids_d = [r[0] for r in con.execute('SELECT DISTINCT s1_id FROM raw_cands_D ORDER BY s1_id').fetchall()]
n_chunks_d = (len(s1_ids_d) + RANK_CHUNK_SIZE - 1) // RANK_CHUNK_SIZE
for ci in range(n_chunks_d):
    chunk = s1_ids_d[ci*RANK_CHUNK_SIZE:(ci+1)*RANK_CHUNK_SIZE]
    con.register('_cids', pd.DataFrame({'s1_id': chunk}))
    con.execute(f"""
    INSERT INTO topk_D
    SELECT s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,
           s1_norm_name,s1_norm_addr,s1_postal,s1_door,t_country,t_raw_name,t_raw_addr,
           t_norm_name,t_norm_addr,t_postal,t_door,jaccard_score,candidate_rank
    FROM (
        SELECT c.s1_id,c.target_id,c.pass_count,s.split,s.country AS s1_country,
               s.raw_name AS s1_raw_name,s.raw_addr AS s1_raw_addr,
               s.norm_name AS s1_norm_name,s.norm_addr AS s1_norm_addr,
               s.postal AS s1_postal,s.door AS s1_door,
               t.country AS t_country,t.raw_name AS t_raw_name,t.raw_addr AS t_raw_addr,
               t.norm_name AS t_norm_name,t.norm_addr AS t_norm_addr,
               t.postal AS t_postal,t.door AS t_door,
               (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0)))
               AS jaccard_score,
               row_number() OVER (PARTITION BY c.s1_id ORDER BY
                   (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                    0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0))) DESC,
                   c.pass_count DESC) AS candidate_rank
        FROM raw_cands_D c
        JOIN _cids x ON c.s1_id=x.s1_id
        JOIN s1_split s ON c.s1_id=s.entity_id
        JOIN target_pool t ON c.target_id=t.entity_id
    ) ranked WHERE candidate_rank <= {K_D};
    """)
tot,s1s,avg,p95 = _cand_stats(con,'topk_D')
r25 = _recall_at_k(con,'topk_D',25,val_gt)
r50 = _recall_at_k(con,'topk_D',50,val_gt)
blocking_results.append({
    'strategy':'D_8pass_perm_K50','K':50,
    'candidate_pairs':tot,'avg_cands':round(avg,1),'p95_cands':p95,
    'recall_at_25':round(r25,4),'recall_at_50':round(r50,4),'recall_at_100':round(r50,4),
    'runtime_s':round(time.time()-t0,1),'peak_mem_mb':round(_mem_mb()-m0,1)
})
print(f'  pairs={tot:,}  avg={avg:.1f}  p95={p95}  recall@25={r25:.4f}  recall@50={r50:.4f}')
gc.collect()


In [ ]:
# ── Strategy E: char 3/4-gram TF-IDF sparse retrieval, K=50 ─────────────
# Memory-safe block-wise top-K retrieval.
# NEVER calls .toarray() on the full Q @ T.T product.
# Processes target matrix in TARGET_BLOCK-row blocks.
# Peak dense allocation per iteration = NGRAM_CHUNK * TARGET_BLOCK floats.
print('Strategy E: char n-gram TF-IDF sparse retrieval (block-wise), K=50...')
t0 = time.time(); m0 = _mem_mb()
K_E = 50
NGRAM_TOP_N   = 30    # top-N per S1 from TF-IDF before merging with SQL passes
NGRAM_CHUNK   = 500   # S1 query rows per inner iteration
TARGET_BLOCK  = 2000  # target rows per block  ->  500*2000 = 1M floats ~ 4MB dense
MIN_SCORE     = 0.05  # discard near-zero cosine matches
FEASIBILITY_LIMIT = 300000  # skip country if target pool exceeds this (OOM risk)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
import scipy.sparse as sp
import heapq, gc

def _sparse_topk_block(Q_norm, T_norm, top_n, min_score, t_block):
    """Block-wise sparse top-K. Never materialises full Q x T^T."""
    n_q = Q_norm.shape[0]
    n_t = T_norm.shape[0]
    heaps = [[] for _ in range(n_q)]
    t_start = 0
    while t_start < n_t:
        t_end    = min(t_start + t_block, n_t)
        T_block  = T_norm[t_start:t_end]
        # Only this small block is converted to dense:
        # shape = (n_q, t_end-t_start), bounded by NGRAM_CHUNK * TARGET_BLOCK
        block_sc = (Q_norm @ T_block.T).toarray()
        for qi in range(n_q):
            row = block_sc[qi]
            for local_ti, sc in enumerate(row):
                if sc <= min_score:
                    continue
                global_ti = t_start + local_ti
                if len(heaps[qi]) < top_n:
                    heapq.heappush(heaps[qi], (sc, global_ti))
                elif sc > heaps[qi][0][0]:
                    heapq.heapreplace(heaps[qi], (sc, global_ti))
        del block_sc
        t_start = t_end
    return heaps

# Create staging table for n-gram candidates
con.execute("""
CREATE OR REPLACE TABLE ngram_cands_E (
    s1_id VARCHAR, target_id VARCHAR, ngram_score FLOAT
);
""")

countries = [r[0] for r in con.execute(
    "SELECT DISTINCT country FROM s1_split ORDER BY country"
).fetchall()]
print(f'  Processing {len(countries)} countries...')

total_ngram_pairs = 0
skipped_countries = []

for ctry in countries:
    t_ctry = time.time()

    tgt_df = con.execute(f"""
        SELECT entity_id, coalesce(norm_name,'') as name
        FROM target_pool WHERE country='{ctry}'
    """).df()
    n_tgt = len(tgt_df)

    if n_tgt < 2:
        del tgt_df; gc.collect(); continue

    # Feasibility gate
    if n_tgt > FEASIBILITY_LIMIT:
        print(f'  [{ctry}] SKIP -- target pool {n_tgt:,} > limit {FEASIBILITY_LIMIT:,}')
        skipped_countries.append({'country': ctry, 'n_tgt': n_tgt, 'reason': 'exceeds_feasibility_limit'})
        del tgt_df; gc.collect(); continue

    s1_df = con.execute(f"""
        SELECT entity_id, coalesce(norm_name,'') as name
        FROM s1_split WHERE country='{ctry}'
    """).df()
    n_s1 = len(s1_df)

    if n_s1 == 0:
        del tgt_df, s1_df; gc.collect(); continue

    vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 4),
                          min_df=1, max_features=50000, sublinear_tf=True)
    try:
        T = vec.fit_transform(tgt_df['name'].tolist())
    except Exception as e:
        print(f'  [{ctry}] TF-IDF fit failed: {e}')
        del tgt_df, s1_df; gc.collect(); continue

    T = normalize(T, norm='l2', copy=False)
    tgt_ids = tgt_df['entity_id'].values
    del tgt_df; gc.collect()

    # Adapt block size if needed to stay under ~200MB dense
    peak_block_mb = NGRAM_CHUNK * TARGET_BLOCK * 4 / 1024 / 1024
    t_block_ctry = TARGET_BLOCK if peak_block_mb <= 200 else max(500, int(200 * 1024 * 1024 / (NGRAM_CHUNK * 4)))

    n_chunks_s1 = (n_s1 + NGRAM_CHUNK - 1) // NGRAM_CHUNK
    ctry_pairs  = 0
    rows_out    = []

    for ci in range(n_chunks_s1):
        s1_chunk = s1_df.iloc[ci * NGRAM_CHUNK : (ci + 1) * NGRAM_CHUNK]
        Q = vec.transform(s1_chunk['name'].tolist())
        Q = normalize(Q, norm='l2', copy=False)

        heaps = _sparse_topk_block(Q, T, NGRAM_TOP_N, MIN_SCORE, t_block_ctry)

        s1_chunk_ids = s1_chunk['entity_id'].values
        for qi, heap in enumerate(heaps):
            for sc, ti in heap:
                rows_out.append((s1_chunk_ids[qi], tgt_ids[ti], float(sc)))

        del Q, heaps
        gc.collect()

        # Flush to DuckDB every 50k rows
        if len(rows_out) >= 50000:
            batch = pd.DataFrame(rows_out, columns=['s1_id', 'target_id', 'ngram_score'])
            con.register('_ng_batch', batch)
            con.execute('INSERT INTO ngram_cands_E SELECT * FROM _ng_batch')
            ctry_pairs += len(rows_out)
            del batch, rows_out; rows_out = []
            gc.collect()

    if rows_out:
        batch = pd.DataFrame(rows_out, columns=['s1_id', 'target_id', 'ngram_score'])
        con.register('_ng_batch', batch)
        con.execute('INSERT INTO ngram_cands_E SELECT * FROM _ng_batch')
        ctry_pairs += len(rows_out)
        del batch, rows_out
        gc.collect()

    total_ngram_pairs += ctry_pairs
    ctry_mem = _mem_mb() - m0
    ctry_rt  = time.time() - t_ctry
    print(f'  [{ctry}] tgt={n_tgt:,}  s1={n_s1:,}  chunk={NGRAM_CHUNK}  '
          f'pairs={ctry_pairs:,}  mem_delta={ctry_mem:+.0f}MB  rt={ctry_rt:.1f}s')

    del s1_df, T, tgt_ids, vec
    gc.collect()

if skipped_countries:
    print(f'\n  Skipped {len(skipped_countries)} countries (feasibility limit):')
    for sc in skipped_countries:
        print(f'    {sc["country"]}  n_tgt={sc["n_tgt"]:,}')

print(f'\n  Total n-gram candidate pairs inserted: {total_ngram_pairs:,}')
print(f'  Peak process memory delta: {_mem_mb()-m0:+.0f}MB')

# Merge n-gram candidates with strategy D raw pairs
con.execute("""
CREATE OR REPLACE TABLE raw_cands_E AS
SELECT s1_id, target_id, max(pass_count) as pass_count
FROM (
    SELECT s1_id, target_id, pass_count FROM raw_cands_D
    UNION ALL
    SELECT s1_id, target_id, 1 as pass_count FROM ngram_cands_E
) sub
GROUP BY s1_id, target_id;
""")
raw_e = con.execute('SELECT count(*) FROM raw_cands_E').fetchone()[0]
print(f'  Raw pairs E (D + n-gram): {raw_e:,}')

# Rank to K_E
con.execute('CREATE OR REPLACE TABLE topk_E (s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,'
            ' split VARCHAR, s1_country VARCHAR,'
            ' s1_raw_name VARCHAR, s1_raw_addr VARCHAR,'
            ' s1_norm_name VARCHAR, s1_norm_addr VARCHAR,'
            ' s1_postal VARCHAR, s1_door VARCHAR,'
            ' t_country VARCHAR, t_raw_name VARCHAR, t_raw_addr VARCHAR,'
            ' t_norm_name VARCHAR, t_norm_addr VARCHAR,'
            ' t_postal VARCHAR, t_door VARCHAR,'
            ' jaccard_score DOUBLE, candidate_rank BIGINT);')
s1_ids_e = [r[0] for r in con.execute('SELECT DISTINCT s1_id FROM raw_cands_E ORDER BY s1_id').fetchall()]
n_chunks_e2 = (len(s1_ids_e) + RANK_CHUNK_SIZE - 1) // RANK_CHUNK_SIZE
for ci in range(n_chunks_e2):
    chunk = s1_ids_e[ci*RANK_CHUNK_SIZE:(ci+1)*RANK_CHUNK_SIZE]
    con.register('_cids', pd.DataFrame({'s1_id': chunk}))
    con.execute(f"""
    INSERT INTO topk_E
    SELECT s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,
           s1_norm_name,s1_norm_addr,s1_postal,s1_door,t_country,t_raw_name,t_raw_addr,
           t_norm_name,t_norm_addr,t_postal,t_door,jaccard_score,candidate_rank
    FROM (
        SELECT c.s1_id,c.target_id,c.pass_count,s.split,s.country AS s1_country,
               s.raw_name AS s1_raw_name,s.raw_addr AS s1_raw_addr,
               s.norm_name AS s1_norm_name,s.norm_addr AS s1_norm_addr,
               s.postal AS s1_postal,s.door AS s1_door,
               t.country AS t_country,t.raw_name AS t_raw_name,t.raw_addr AS t_raw_addr,
               t.norm_name AS t_norm_name,t.norm_addr AS t_norm_addr,
               t.postal AS t_postal,t.door AS t_door,
               (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0)))
               AS jaccard_score,
               row_number() OVER (PARTITION BY c.s1_id ORDER BY
                   (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                    0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0))) DESC,
                   c.pass_count DESC) AS candidate_rank
        FROM raw_cands_E c
        JOIN _cids x ON c.s1_id=x.s1_id
        JOIN s1_split s ON c.s1_id=s.entity_id
        JOIN target_pool t ON c.target_id=t.entity_id
    ) ranked WHERE candidate_rank <= {K_E};
    """)
tot,s1s,avg,p95 = _cand_stats(con,'topk_E')
r25 = _recall_at_k(con,'topk_E',25,val_gt)
r50 = _recall_at_k(con,'topk_E',50,val_gt)
blocking_results.append({
    'strategy':'E_8pass_perm_ngram_K50','K':50,
    'candidate_pairs':tot,'avg_cands':round(avg,1),'p95_cands':p95,
    'recall_at_25':round(r25,4),'recall_at_50':round(r50,4),'recall_at_100':round(r50,4),
    'runtime_s':round(time.time()-t0,1),'peak_mem_mb':round(_mem_mb()-m0,1)
})
print(f'  pairs={tot:,}  avg={avg:.1f}  p95={p95}  recall@25={r25:.4f}  recall@50={r50:.4f}')
gc.collect()


In [ ]:
# ── Strategy F: 8-pass + perm + ngram + extra address passes, K=50 ───────
# Extra address passes: a1+a2, postal+a1, door+postal+country
print('Strategy F: full combined blocking, K=50...')
t0 = time.time(); m0 = _mem_mb()
K_F = 50
con.execute("""
CREATE OR REPLACE TABLE raw_cands_F AS
SELECT s1_id, target_id, max(pass_count) as pass_count
FROM (
    SELECT s1_id, target_id, pass_count FROM raw_cands_E
    -- extra address pass 1: a1+a2 (first two address tokens)
    UNION ALL
    SELECT s.entity_id,t.entity_id,1 FROM s1_split s JOIN target_pool t
        ON s.country=t.country AND s.a1=t.a1 AND s.a2=t.a2
        WHERE length(s.a1)>=4 AND length(s.a2)>=4
    -- extra address pass 2: postal+a1
    UNION ALL
    SELECT s.entity_id,t.entity_id,1 FROM s1_split s JOIN target_pool t
        ON s.country=t.country AND s.postal=t.postal AND s.a1=t.a1
        WHERE s.postal!='' AND length(s.a1)>=4
    -- extra address pass 3: door+postal (without requiring name token)
    UNION ALL
    SELECT s.entity_id,t.entity_id,1 FROM s1_split s JOIN target_pool t
        ON s.country=t.country AND s.door=t.door AND s.postal=t.postal
        WHERE length(s.door)>=1 AND s.postal!=''
) sub
GROUP BY s1_id, target_id;
""")
raw_f = con.execute('SELECT count(*) FROM raw_cands_F').fetchone()[0]
print(f'  Raw pairs F: {raw_f:,}')

con.execute('CREATE OR REPLACE TABLE topk_F (s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,'
            ' split VARCHAR, s1_country VARCHAR,'
            ' s1_raw_name VARCHAR, s1_raw_addr VARCHAR,'
            ' s1_norm_name VARCHAR, s1_norm_addr VARCHAR,'
            ' s1_postal VARCHAR, s1_door VARCHAR,'
            ' t_country VARCHAR, t_raw_name VARCHAR, t_raw_addr VARCHAR,'
            ' t_norm_name VARCHAR, t_norm_addr VARCHAR,'
            ' t_postal VARCHAR, t_door VARCHAR,'
            ' jaccard_score DOUBLE, candidate_rank BIGINT);')
s1_ids_f = [r[0] for r in con.execute('SELECT DISTINCT s1_id FROM raw_cands_F ORDER BY s1_id').fetchall()]
n_chunks_f = (len(s1_ids_f) + RANK_CHUNK_SIZE - 1) // RANK_CHUNK_SIZE
for ci in range(n_chunks_f):
    chunk = s1_ids_f[ci*RANK_CHUNK_SIZE:(ci+1)*RANK_CHUNK_SIZE]
    con.register('_cids', pd.DataFrame({'s1_id': chunk}))
    con.execute(f"""
    INSERT INTO topk_F
    SELECT s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,
           s1_norm_name,s1_norm_addr,s1_postal,s1_door,t_country,t_raw_name,t_raw_addr,
           t_norm_name,t_norm_addr,t_postal,t_door,jaccard_score,candidate_rank
    FROM (
        SELECT c.s1_id,c.target_id,c.pass_count,s.split,s.country AS s1_country,
               s.raw_name AS s1_raw_name,s.raw_addr AS s1_raw_addr,
               s.norm_name AS s1_norm_name,s.norm_addr AS s1_norm_addr,
               s.postal AS s1_postal,s.door AS s1_door,
               t.country AS t_country,t.raw_name AS t_raw_name,t.raw_addr AS t_raw_addr,
               t.norm_name AS t_norm_name,t.norm_addr AS t_norm_addr,
               t.postal AS t_postal,t.door AS t_door,
               (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                     nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0)))
               AS jaccard_score,
               row_number() OVER (PARTITION BY c.s1_id ORDER BY
                   (0.6*(len(list_intersect(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_name,' '),string_split(t.norm_name,' ')))),0))+
                    0.4*(len(list_intersect(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))::float/
                         nullif(len(list_distinct(list_concat(string_split(s.norm_addr,' '),string_split(t.norm_addr,' ')))),0))) DESC,
                   c.pass_count DESC) AS candidate_rank
        FROM raw_cands_F c
        JOIN _cids x ON c.s1_id=x.s1_id
        JOIN s1_split s ON c.s1_id=s.entity_id
        JOIN target_pool t ON c.target_id=t.entity_id
    ) ranked WHERE candidate_rank <= {K_F};
    """)
tot,s1s,avg,p95 = _cand_stats(con,'topk_F')
r25 = _recall_at_k(con,'topk_F',25,val_gt)
r50 = _recall_at_k(con,'topk_F',50,val_gt)
blocking_results.append({
    'strategy':'F_full_combined_K50','K':50,
    'candidate_pairs':tot,'avg_cands':round(avg,1),'p95_cands':p95,
    'recall_at_25':round(r25,4),'recall_at_50':round(r50,4),'recall_at_100':round(r50,4),
    'runtime_s':round(time.time()-t0,1),'peak_mem_mb':round(_mem_mb()-m0,1)
})
print(f'  pairs={tot:,}  avg={avg:.1f}  p95={p95}  recall@25={r25:.4f}  recall@50={r50:.4f}')
gc.collect()

# ── BLOCKING COMPARISON TABLE ─────────────────────────────────────────────
print('\n' + '='*80)
print('BLOCKING STRATEGY COMPARISON')
print('='*80)
df_blocking = pd.DataFrame(blocking_results)
display(df_blocking[['strategy','K','candidate_pairs','avg_cands','p95_cands',
                      'recall_at_25','recall_at_50','recall_at_100','runtime_s','peak_mem_mb']])

# Save blocking comparison
os.makedirs('/content/experiment_v2', exist_ok=True)
df_blocking.to_csv('/content/experiment_v2/blocking_comparison.csv', index=False)
print('Saved: /content/experiment_v2/blocking_comparison.csv')

# Select best strategy by recall@50 (primary) then candidate_pairs (secondary — prefer fewer)
best_blocking = df_blocking.sort_values(
    ['recall_at_50','candidate_pairs'], ascending=[False, True]
).iloc[0]
BEST_STRATEGY = best_blocking['strategy']
BEST_TOPK_TABLE = {
    'A_8pass_K25':       'topk_candidates',
    'B_8pass_K50':       'topk_B',
    'C_8pass_K100':      'topk_C',
    'D_8pass_perm_K50':  'topk_D',
    'E_8pass_perm_ngram_K50': 'topk_E',
    'F_full_combined_K50':    'topk_F',
}.get(BEST_STRATEGY, 'topk_F')
print(f'\nBest blocking strategy: {BEST_STRATEGY}  (table: {BEST_TOPK_TABLE})')
print(f'  recall@50={best_blocking["recall_at_50"]:.4f}  avg_cands={best_blocking["avg_cands"]}')


### Section 22: Improved Feature Extraction (v2) — Best Blocking Table

Reuses the existing 24-feature pipeline and adds 6 candidate-context features:
- `norm_rank` — rank / K (normalized)
- `score_margin` — jaccard_score - top_jaccard_score for this S1
- `top_score` — jaccard_score of rank-1 candidate for this S1
- `name_addr_composite` — 0.7*name_sort + 0.3*addr_sort
- `rank_x_jaccard` — candidate_rank * jaccard_score interaction
- `pass_count_norm` — pass_count / max_pass_count (normalized)

Total: **30 features**. No target leakage — context features use only blocking scores.


In [ ]:
import gc
from rapidfuzz import fuzz as _fuzz2, distance as _dist2

FEATURE_NAMES_V2 = [
    # original 24
    'name_exact_raw','name_exact_norm','name_token_sort_ratio','name_token_set_ratio',
    'name_jaro_winkler','name_levenshtein_ratio','name_char_sim','name_substring_containment',
    'addr_exact_norm','addr_token_sort_ratio','addr_token_set_ratio','addr_token_jaccard',
    'addr_char_sim','addr_num_overlap','addr_postal_match',
    'country_agreement','s1_name_missing','t_name_missing','s1_addr_missing','t_addr_missing',
    'candidate_rank','blocking_pass_count','name_x_addr','name_diff_addr',
    # 6 new context features
    'norm_rank','score_margin','top_score','name_addr_composite',
    'rank_x_jaccard','pass_count_norm',
]
N_FEAT_V2 = len(FEATURE_NAMES_V2)
FEATURE_CHUNK_SIZE = 50000

def compute_pair_features_v2(s1_raw,s1_norm,s1_addr,s1_postal,s1_door,
                              t_raw,t_norm,t_addr,t_postal,t_door,
                              s1_country,t_country,rank,pass_count,
                              jaccard_score,top_jac,k_val,max_pass):
    # original 24 features
    exact_raw  = 1.0 if s1_raw and s1_raw==t_raw else 0.0
    exact_norm = 1.0 if s1_norm and s1_norm==t_norm else 0.0
    name_sort  = _fuzz2.token_sort_ratio(s1_norm,t_norm)/100.0
    name_set   = _fuzz2.token_set_ratio(s1_norm,t_norm)/100.0
    jw         = _dist2.JaroWinkler.similarity(s1_norm,t_norm)
    lev        = _dist2.Levenshtein.normalized_similarity(s1_norm,t_norm)
    name_char  = _fuzz2.ratio(s1_norm,t_norm)/100.0
    name_sub   = 1.0 if (s1_norm and t_norm and (s1_norm in t_norm or t_norm in s1_norm)) else 0.0
    addr_exact = 1.0 if s1_addr and s1_addr==t_addr else 0.0
    addr_sort  = _fuzz2.token_sort_ratio(s1_addr,t_addr)/100.0
    addr_set   = _fuzz2.token_set_ratio(s1_addr,t_addr)/100.0
    s1_at = set(s1_addr.split()) if s1_addr else set()
    t_at  = set(t_addr.split())  if t_addr  else set()
    addr_jac   = len(s1_at&t_at)/len(s1_at|t_at) if (s1_at or t_at) else 0.0
    addr_char  = _fuzz2.ratio(s1_addr,t_addr)/100.0
    s1_nums = {w for w in s1_at if w.isdigit()}
    t_nums  = {w for w in t_at  if w.isdigit()}
    if s1_nums and t_nums:           num_ov = len(s1_nums&t_nums)/len(s1_nums|t_nums)
    elif not s1_nums and not t_nums: num_ov = 0.5
    else:                            num_ov = 0.0
    postal_m   = 1.0 if s1_postal and t_postal and s1_postal==t_postal else 0.0
    country_ag = 1.0 if s1_country==t_country else 0.0
    s1nm = 1.0 if not s1_norm else 0.0
    tnm  = 1.0 if not t_norm  else 0.0
    s1am = 1.0 if not s1_addr else 0.0
    tam  = 1.0 if not t_addr  else 0.0
    rank_f = 1.0/rank if rank>0 else 0.0
    pc_f   = float(pass_count)
    nxa    = name_sort*addr_sort
    nda    = abs(name_sort-addr_sort)
    # 6 new context features
    norm_rank_f   = rank / max(k_val, 1)
    score_margin  = float(jaccard_score) - float(top_jac) if top_jac is not None else 0.0
    top_score_f   = float(top_jac) if top_jac is not None else 0.0
    composite     = 0.7*name_sort + 0.3*addr_sort
    rank_x_jac    = rank_f * float(jaccard_score)
    pc_norm       = float(pass_count) / max(float(max_pass), 1.0)
    return [exact_raw,exact_norm,name_sort,name_set,jw,lev,name_char,name_sub,
            addr_exact,addr_sort,addr_set,addr_jac,addr_char,num_ov,postal_m,
            country_ag,s1nm,tnm,s1am,tam,rank_f,pc_f,nxa,nda,
            norm_rank_f,score_margin,top_score_f,composite,rank_x_jac,pc_norm]

def _fill_X_v2(df_chunk, X, y, row_offset, k_val, top_jac_map, max_pass):
    n = len(df_chunk)
    s1_raw  = df_chunk['s1_raw_name'].fillna('').values
    s1_norm = df_chunk['s1_norm_name'].fillna('').values
    s1_addr = df_chunk['s1_norm_addr'].fillna('').values
    s1_post = df_chunk['s1_postal'].fillna('').values
    s1_door = df_chunk['s1_door'].fillna('').values
    s1_coun = df_chunk['s1_country'].fillna('').values
    t_raw   = df_chunk['t_raw_name'].fillna('').values
    t_norm  = df_chunk['t_norm_name'].fillna('').values
    t_addr  = df_chunk['t_norm_addr'].fillna('').values
    t_post  = df_chunk['t_postal'].fillna('').values
    t_door  = df_chunk['t_door'].fillna('').values
    t_coun  = df_chunk['t_country'].fillna('').values
    ranks   = df_chunk['candidate_rank'].values
    passes  = df_chunk['pass_count'].values
    jacs    = df_chunk['jaccard_score'].fillna(0).values
    labels  = df_chunk['label'].values
    s1_ids  = df_chunk['s1_id'].values
    for i in range(n):
        sid = s1_ids[i]
        top_jac = top_jac_map.get(sid, 0.0)
        X[row_offset+i] = compute_pair_features_v2(
            str(s1_raw[i] or ''), str(s1_norm[i] or ''), str(s1_addr[i] or ''),
            str(s1_post[i] or ''), str(s1_door[i] or ''),
            str(t_raw[i] or ''),  str(t_norm[i] or ''),  str(t_addr[i] or ''),
            str(t_post[i] or ''), str(t_door[i] or ''),
            str(s1_coun[i] or ''), str(t_coun[i] or ''),
            int(ranks[i] or 1), int(passes[i] or 1),
            float(jacs[i]), top_jac, k_val, max_pass
        )
        y[row_offset+i] = int(labels[i])

# ── Determine K and max_pass for best blocking table ──────────────────────
K_BEST = int(best_blocking['K'])
max_pass_val = con.execute(f'SELECT max(pass_count) FROM {BEST_TOPK_TABLE}').fetchone()[0] or 1

# Build top-jaccard map per S1 (rank=1 score) — small aggregation, safe
top_jac_rows = con.execute(f"""
    SELECT s1_id, jaccard_score FROM {BEST_TOPK_TABLE}
    WHERE candidate_rank=1
""").fetchall()
top_jac_map = {r[0]: float(r[1]) for r in top_jac_rows}
print(f'Top-jaccard map built: {len(top_jac_map):,} S1 entries')

# ── Label best-blocking candidates ────────────────────────────────────────
con.execute(f"""
CREATE OR REPLACE TABLE labeled_best AS
SELECT c.*,
    case when g.target_id is not null then 1 else 0 end as label
FROM {BEST_TOPK_TABLE} c
LEFT JOIN (
    SELECT entity_id as s1_id, unnest(string_split(matched_entity_ids,',')) as target_id
    FROM s1_gt WHERE matched_entity_ids is not null
) g ON c.s1_id=g.s1_id AND c.target_id=g.target_id;
""")

# ── Hard negatives for best blocking (reuse same 3-pass approach) ─────────
print('Building hard negatives for best blocking table...')
from rapidfuzz import fuzz as _fuzz
import numpy as _np2

CATEGORIES = [
    'high_name_low_addr','high_addr_low_name','high_name_conflict_door',
    'high_addr_diff_name','generic_name_collision','remaining_difficult',
]
def _assign_category(ns, as_, s1d, td, s1n):
    if ns >= 80 and as_ < 40:                        return 'high_name_low_addr'
    elif as_ >= 75 and ns < 40:                      return 'high_addr_low_name'
    elif ns >= 70 and s1d and td and s1d != td:      return 'high_name_conflict_door'
    elif as_ >= 70 and ns < 60:                      return 'high_addr_diff_name'
    elif len(s1n.replace(' ','')) <= 5:              return 'generic_name_collision'
    else:                                            return 'remaining_difficult'

train_pos_v2 = con.execute("SELECT count(*) FROM labeled_best WHERE split='train' AND label=1").fetchone()[0]
total_neg_v2 = con.execute("SELECT count(*) FROM labeled_best WHERE split='train' AND label=0").fetchone()[0]
max_negs_v2  = int(train_pos_v2 * HARD_NEG_RATIO)
HARD_NEG_CHUNK_SIZE = 50000
NEG_COLS_V2 = ('s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,'
               's1_norm_name,s1_norm_addr,s1_postal,s1_door,'
               't_country,t_raw_name,t_raw_addr,t_norm_name,t_norm_addr,t_postal,t_door,'
               'jaccard_score,candidate_rank')
NEG_Q_V2 = f"SELECT {NEG_COLS_V2} FROM labeled_best WHERE split='train' AND label=0"
n_neg_chunks_v2 = (total_neg_v2 + HARD_NEG_CHUNK_SIZE - 1) // HARD_NEG_CHUNK_SIZE

# Pass 1: count
cat_counts_v2 = {c: 0 for c in CATEGORIES}
for ci in range(n_neg_chunks_v2):
    chunk = con.execute(f'{NEG_Q_V2} LIMIT {HARD_NEG_CHUNK_SIZE} OFFSET {ci*HARD_NEG_CHUNK_SIZE}').df()
    for i in range(len(chunk)):
        ns  = _fuzz.token_sort_ratio(str(chunk['s1_norm_name'].iloc[i] or ''), str(chunk['t_norm_name'].iloc[i] or ''))
        as_ = _fuzz.token_sort_ratio(str(chunk['s1_norm_addr'].iloc[i] or ''), str(chunk['t_norm_addr'].iloc[i] or ''))
        cat_counts_v2[_assign_category(ns, as_, str(chunk['s1_door'].iloc[i] or ''), str(chunk['t_door'].iloc[i] or ''), str(chunk['s1_norm_name'].iloc[i] or ''))] += 1
    del chunk; gc.collect()

# Pass 2: targets
cat_targets_v2 = {}; remaining_v2 = max_negs_v2
cats_with_data_v2 = [c for c in CATEGORIES if cat_counts_v2[c] > 0]
for idx, cat in enumerate(cats_with_data_v2):
    share = remaining_v2 if idx == len(cats_with_data_v2)-1 else int(max_negs_v2 * cat_counts_v2[cat] / max(total_neg_v2,1))
    share = min(share, cat_counts_v2[cat], remaining_v2)
    cat_targets_v2[cat] = share; remaining_v2 -= share

# Pass 3: sample
con.execute(f"""
CREATE OR REPLACE TABLE hard_neg_v2 (
    s1_id VARCHAR, target_id VARCHAR, pass_count BIGINT,
    split VARCHAR, s1_country VARCHAR,
    s1_raw_name VARCHAR, s1_raw_addr VARCHAR,
    s1_norm_name VARCHAR, s1_norm_addr VARCHAR,
    s1_postal VARCHAR, s1_door VARCHAR,
    t_country VARCHAR, t_raw_name VARCHAR, t_raw_addr VARCHAR,
    t_norm_name VARCHAR, t_norm_addr VARCHAR,
    t_postal VARCHAR, t_door VARCHAR,
    jaccard_score DOUBLE, candidate_rank BIGINT
);
""")
cat_filled_v2 = {c: 0 for c in CATEGORIES}
rng_v2 = _np2.random.default_rng(SEED)
import pandas as _pd_v2
for ci in range(n_neg_chunks_v2):
    chunk = con.execute(f'{NEG_Q_V2} LIMIT {HARD_NEG_CHUNK_SIZE} OFFSET {ci*HARD_NEG_CHUNK_SIZE}').df()
    chunk_cats = []
    for i in range(len(chunk)):
        ns  = _fuzz.token_sort_ratio(str(chunk['s1_norm_name'].iloc[i] or ''), str(chunk['t_norm_name'].iloc[i] or ''))
        as_ = _fuzz.token_sort_ratio(str(chunk['s1_norm_addr'].iloc[i] or ''), str(chunk['t_norm_addr'].iloc[i] or ''))
        chunk_cats.append(_assign_category(ns, as_, str(chunk['s1_door'].iloc[i] or ''), str(chunk['t_door'].iloc[i] or ''), str(chunk['s1_norm_name'].iloc[i] or '')))
    chunk['_cat'] = chunk_cats
    rows_to_insert = []
    for cat in CATEGORIES:
        need = cat_targets_v2.get(cat,0) - cat_filled_v2[cat]
        if need <= 0: continue
        cat_rows = chunk[chunk['_cat']==cat]
        if len(cat_rows) == 0: continue
        share = min(need, max(1, int(cat_targets_v2.get(cat,0)*len(cat_rows)/max(cat_counts_v2[cat],1))), len(cat_rows), need)
        if share > 0:
            sampled = cat_rows.sample(n=share, random_state=int(rng_v2.integers(0,2**31)))
            rows_to_insert.append(sampled.drop(columns=['_cat']))
            cat_filled_v2[cat] += share
    if rows_to_insert:
        batch = _pd_v2.concat(rows_to_insert, ignore_index=True)
        con.register('_neg_v2', batch)
        con.execute('INSERT INTO hard_neg_v2 SELECT * FROM _neg_v2')
        del batch
    del chunk, rows_to_insert; gc.collect()

final_neg_v2 = con.execute('SELECT count(*) FROM hard_neg_v2').fetchone()[0]
print(f'Hard negatives v2: {final_neg_v2:,}  pos: {train_pos_v2:,}  ratio: {final_neg_v2/max(train_pos_v2,1):.2f}')

con.execute("""
CREATE OR REPLACE TABLE train_v2 AS
SELECT s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,
       s1_norm_name,s1_norm_addr,s1_postal,s1_door,
       t_country,t_raw_name,t_raw_addr,t_norm_name,t_norm_addr,t_postal,t_door,
       jaccard_score,candidate_rank,1 as label
FROM labeled_best WHERE split='train' AND label=1
UNION ALL
SELECT s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,
       s1_norm_name,s1_norm_addr,s1_postal,s1_door,
       t_country,t_raw_name,t_raw_addr,t_norm_name,t_norm_addr,t_postal,t_door,
       jaccard_score,candidate_rank,0 as label
FROM hard_neg_v2;
""")

train_v2_cnt = con.execute('SELECT count(*) FROM train_v2').fetchone()[0]
val_v2_cnt   = con.execute("SELECT count(*) FROM labeled_best WHERE split='val'").fetchone()[0]
print(f'train_v2: {train_v2_cnt:,}  val_v2: {val_v2_cnt:,}')

# ── Chunked feature extraction ─────────────────────────────────────────────
print(f'Extracting features v2 ({N_FEAT_V2} features)...')
t_feat2 = time.time()
COLS_V2 = ('s1_id,target_id,pass_count,split,s1_country,s1_raw_name,s1_raw_addr,'
           's1_norm_name,s1_norm_addr,s1_postal,s1_door,'
           't_country,t_raw_name,t_raw_addr,t_norm_name,t_norm_addr,t_postal,t_door,'
           'jaccard_score,candidate_rank,label')

X_train_v2 = _np2.empty((train_v2_cnt, N_FEAT_V2), dtype=_np2.float32)
y_train_v2 = _np2.empty(train_v2_cnt, dtype=_np2.int32)
offset = 0
n_tr_chunks = (train_v2_cnt + FEATURE_CHUNK_SIZE - 1) // FEATURE_CHUNK_SIZE
for ci in range(n_tr_chunks):
    chunk_df = con.execute(f'SELECT {COLS_V2} FROM train_v2 LIMIT {FEATURE_CHUNK_SIZE} OFFSET {ci*FEATURE_CHUNK_SIZE}').df()
    _fill_X_v2(chunk_df, X_train_v2, y_train_v2, offset, K_BEST, top_jac_map, max_pass_val)
    offset += len(chunk_df)
    del chunk_df; gc.collect()
    if (ci+1) % 5 == 0 or ci == n_tr_chunks-1:
        print(f'  train chunk {ci+1}/{n_tr_chunks}  rows={offset:,}')

X_val_v2   = _np2.empty((val_v2_cnt, N_FEAT_V2), dtype=_np2.float32)
y_val_v2   = _np2.empty(val_v2_cnt, dtype=_np2.int32)
val_s1_ids_v2  = _np2.empty(val_v2_cnt, dtype=object)
val_tgt_ids_v2 = _np2.empty(val_v2_cnt, dtype=object)
offset = 0
n_vl_chunks = (val_v2_cnt + FEATURE_CHUNK_SIZE - 1) // FEATURE_CHUNK_SIZE
for ci in range(n_vl_chunks):
    chunk_df = con.execute(f"SELECT {COLS_V2} FROM labeled_best WHERE split='val' LIMIT {FEATURE_CHUNK_SIZE} OFFSET {ci*FEATURE_CHUNK_SIZE}").df()
    _fill_X_v2(chunk_df, X_val_v2, y_val_v2, offset, K_BEST, top_jac_map, max_pass_val)
    val_s1_ids_v2 [offset:offset+len(chunk_df)] = chunk_df['s1_id'].values
    val_tgt_ids_v2[offset:offset+len(chunk_df)] = chunk_df['target_id'].values
    offset += len(chunk_df)
    del chunk_df; gc.collect()
    if (ci+1) % 5 == 0 or ci == n_vl_chunks-1:
        print(f'  val chunk {ci+1}/{n_vl_chunks}  rows={offset:,}')

print(f'Feature extraction v2 done in {time.time()-t_feat2:.1f}s')
print(f'  X_train_v2: {X_train_v2.shape}  X_val_v2: {X_val_v2.shape}')
nan_v2 = int(_np2.isnan(X_train_v2).sum())
inf_v2 = int(_np2.isinf(X_train_v2).sum())
print(f'  NaN: {nan_v2}  Inf: {inf_v2}')
if nan_v2 > 0 or inf_v2 > 0:
    raise ValueError('NaN/Inf in X_train_v2!')
print('  [PASS] Feature validation passed.')


### Section 23: Model Training v2 — XGBoost + CatBoost

Train on the improved feature set (30 features) from the best blocking table.
Same train/val split as baseline. Conservative memory settings throughout.


In [ ]:
import xgboost as xgb
import numpy as _np2

# ── XGBoost v2 ────────────────────────────────────────────────────────────
print('Training XGBoost v2 (30 features, best blocking)...')
t_xgb2 = time.time()
xgb_model_v2 = xgb.XGBClassifier(
    n_estimators=400, max_depth=7, learning_rate=0.07,
    subsample=0.8, colsample_bytree=0.8,
    tree_method='hist', random_state=SEED, n_jobs=4
)
xgb_model_v2.fit(X_train_v2, y_train_v2)
xgb_time_v2 = time.time() - t_xgb2
xgb_probs_v2 = xgb_model_v2.predict_proba(X_val_v2)[:,1]
print(f'  XGBoost v2 trained in {xgb_time_v2:.2f}s')

imp_v2 = xgb_model_v2.feature_importances_
top_idx_v2 = _np2.argsort(imp_v2)[::-1][:10]
print('Top 10 XGBoost v2 Feature Importances:')
for r, idx in enumerate(top_idx_v2, 1):
    print(f'  {r:>2}. {FEATURE_NAMES_V2[idx]:<30}: {imp_v2[idx]*100:5.2f}%')

# ── CatBoost ──────────────────────────────────────────────────────────────
print('\nTraining CatBoost...')
t_cb = time.time()
cb_model = CatBoostClassifier(
    iterations=400, depth=7, learning_rate=0.07,
    loss_function='Logloss', eval_metric='AUC',
    verbose=False, random_seed=SEED, thread_count=4,
    task_type='CPU'
)
cb_model.fit(X_train_v2, y_train_v2)
cb_time = time.time() - t_cb
cb_probs = cb_model.predict_proba(X_val_v2)[:,1]
print(f'  CatBoost trained in {cb_time:.2f}s')

cb_imp = cb_model.get_feature_importance()
top_cb = _np2.argsort(cb_imp)[::-1][:10]
print('Top 10 CatBoost Feature Importances:')
for r, idx in enumerate(top_cb, 1):
    print(f'  {r:>2}. {FEATURE_NAMES_V2[idx]:<30}: {cb_imp[idx]:5.2f}')


### Section 24: Ensemble + Threshold Search (All v2 Models)

Ensemble: `alpha * xgb_prob + (1-alpha) * cb_prob` for alpha in {0.25, 0.50, 0.75}.
Threshold grid: 0.30–0.95 step 0.05. Metric: Macro F0.5 (same implementation as baseline).
All models evaluated on the same val S1 entities.


In [ ]:
from collections import defaultdict
import numpy as _np2
import pandas as _pd_v2

# Build val GT map for v2 (same val_gt dict already exists from Section 13)
# val_gt is already built — reuse it

def evaluate_threshold_grid_v2(val_probs, model_name, s1_ids, tgt_ids):
    scores_by_s1 = defaultdict(list)
    for s1_id, tgt_id, prob in zip(s1_ids, tgt_ids, val_probs):
        scores_by_s1[s1_id].append((tgt_id, float(prob)))
    thresholds = [round(t,2) for t in _np2.arange(0.30, 0.96, 0.05)]
    results, best_row, best_f05 = [], None, -1.0
    for tau in thresholds:
        f05_list = []
        total_tp = total_fp = total_fn = total_preds = 0
        for sid, true_set in val_gt.items():
            cands    = scores_by_s1.get(sid, [])
            pred_set = {tid for tid, prob in cands if prob >= tau}
            f05_list.append(compute_entity_f05(true_set, pred_set))
            total_tp    += len(true_set & pred_set)
            total_fp    += len(pred_set - true_set)
            total_fn    += len(true_set - pred_set)
            total_preds += len(pred_set)
        macro_f05 = float(_np2.mean(f05_list))
        prec = total_tp/(total_tp+total_fp) if (total_tp+total_fp)>0 else 0.0
        rec  = total_tp/(total_tp+total_fn) if (total_tp+total_fn)>0 else 0.0
        row  = {'model':model_name,'threshold':tau,
                'macro_f05':round(macro_f05,4),
                'precision':round(prec,4),'recall':round(rec,4),
                'false_positives':total_fp,'false_negatives':total_fn,
                'avg_matches_per_s1':round(total_preds/len(val_gt),3),
                'n_val_s1_evaluated':len(val_gt)}
        results.append(row)
        if macro_f05 > best_f05:
            best_f05, best_row = macro_f05, row
    return _pd_v2.DataFrame(results), best_row

# ── Ensemble probabilities ─────────────────────────────────────────────────
ens_25 = 0.25 * xgb_probs_v2 + 0.75 * cb_probs
ens_50 = 0.50 * xgb_probs_v2 + 0.50 * cb_probs
ens_75 = 0.75 * xgb_probs_v2 + 0.25 * cb_probs

# ── Threshold search for all models ───────────────────────────────────────
print('Running threshold search for all v2 models...')
models_v2 = [
    ('XGBoost_v2',          xgb_probs_v2),
    ('CatBoost',            cb_probs),
    ('Ensemble_XGB25_CB75', ens_25),
    ('Ensemble_XGB50_CB50', ens_50),
    ('Ensemble_XGB75_CB25', ens_75),
]

all_thresh_dfs_v2 = []
best_rows_v2 = {}
for mname, mprobs in models_v2:
    df_t, best_r = evaluate_threshold_grid_v2(mprobs, mname, val_s1_ids_v2, val_tgt_ids_v2)
    all_thresh_dfs_v2.append(df_t)
    best_rows_v2[mname] = best_r
    print(f'  {mname:<30}  F0.5={best_r["macro_f05"]:.4f}  tau={best_r["threshold"]:.2f}  P={best_r["precision"]:.4f}  R={best_r["recall"]:.4f}')

thresh_df_v2 = _pd_v2.concat(all_thresh_dfs_v2, ignore_index=True)
thresh_df_v2.to_csv('/content/experiment_v2/threshold_results_v2.csv', index=False)
print('Saved: /content/experiment_v2/threshold_results_v2.csv')


### Section 25: Fair Comparison Table

All models evaluated on the same 30,001 validation S1 entities with the same Macro F0.5 implementation.


In [ ]:
import pandas as _pd_v2
from sklearn.metrics import roc_auc_score

# Baseline results (from Section 13/14)
baseline_row = {
    'Experiment': 'Baseline_8pass_K25',
    'Blocking': '8-pass K=25',
    'Candidate_Recall_at_K': blocking_results[0]['recall_at_25'],
    'Model': 'XGBoost',
    'Macro_F05': xgb_best['macro_f05'],
    'Precision': xgb_best['precision'],
    'Recall': xgb_best['recall'],
    'Threshold': xgb_best['threshold'],
    'Train_Time_s': xgb_time,
    'ROC_AUC': round(roc_auc_score(y_val, xgb_val_probs), 4),
}

# Best blocking recall from comparison
best_block_recall = best_blocking['recall_at_50']

# v2 rows
v2_rows = []
for mname, _ in models_v2:
    br = best_rows_v2[mname]
    v2_rows.append({
        'Experiment': f'v2_{BEST_STRATEGY}',
        'Blocking': f'{BEST_STRATEGY} K={K_BEST}',
        'Candidate_Recall_at_K': best_block_recall,
        'Model': mname,
        'Macro_F05': br['macro_f05'],
        'Precision': br['precision'],
        'Recall': br['recall'],
        'Threshold': br['threshold'],
        'Train_Time_s': (
            xgb_time_v2 if 'XGBoost' in mname
            else cb_time if 'CatBoost' in mname
            else xgb_time_v2 + cb_time
        ),
        'ROC_AUC': round(roc_auc_score(y_val_v2, (
            xgb_probs_v2 if 'XGBoost' in mname
            else cb_probs if 'CatBoost' in mname
            else ens_50
        )), 4),
    })

comparison_df = _pd_v2.DataFrame([baseline_row] + v2_rows)
print('=' * 100)
print('FINAL EXPERIMENT COMPARISON TABLE')
print('=' * 100)
display(comparison_df[['Experiment','Blocking','Candidate_Recall_at_K','Model',
                        'Macro_F05','Precision','Recall','Threshold','Train_Time_s','ROC_AUC']])

comparison_df.to_csv('/content/experiment_v2/model_comparison_v2.csv', index=False)
print('Saved: /content/experiment_v2/model_comparison_v2.csv')

# Identify best overall model
best_overall = comparison_df.loc[comparison_df['Macro_F05'].idxmax()]
print(f'\nBest overall: {best_overall["Model"]} ({best_overall["Experiment"]})')
print(f'  Macro F0.5 = {best_overall["Macro_F05"]}')
print(f'  Precision  = {best_overall["Precision"]}')
print(f'  Recall     = {best_overall["Recall"]}')
print(f'  Threshold  = {best_overall["Threshold"]}')

# Determine best v2 model name and probs for downstream use
best_v2_name = max(best_rows_v2, key=lambda k: best_rows_v2[k]['macro_f05'])
best_v2_probs = {
    'XGBoost_v2':          xgb_probs_v2,
    'CatBoost':            cb_probs,
    'Ensemble_XGB25_CB75': ens_25,
    'Ensemble_XGB50_CB50': ens_50,
    'Ensemble_XGB75_CB25': ens_75,
}[best_v2_name]
best_v2_tau = best_rows_v2[best_v2_name]['threshold']
print(f'\nBest v2 model: {best_v2_name}  tau*={best_v2_tau}')


### Section 26: Error Analysis v2

Analyzes FP/FN for the best v2 model and blocking misses (true matches never in candidate pool).


In [ ]:
import gc
from collections import defaultdict
from rapidfuzz import fuzz as _fuzz
import pandas as _pd_v2
import numpy as _np2

pred_v2 = (best_v2_probs >= best_v2_tau).astype(_np2.int32)
fp_mask_v2 = (y_val_v2 == 0) & (pred_v2 == 1)
fn_mask_v2 = (y_val_v2 == 1) & (pred_v2 == 0)
print(f'Best v2 model: {best_v2_name}  tau*={best_v2_tau}')
print(f'False Positives: {fp_mask_v2.sum():,}  False Negatives: {fn_mask_v2.sum():,}')

# ── FALSE POSITIVE ANALYSIS ────────────────────────────────────────────────
fp_s1_ids_v2  = val_s1_ids_v2[fp_mask_v2].tolist()
fp_tgt_ids_v2 = val_tgt_ids_v2[fp_mask_v2].tolist()

fp_cats_v2 = defaultdict(int)
if fp_s1_ids_v2:
    con.register('_fp_s1_v2',  _pd_v2.DataFrame({'s1_id':  fp_s1_ids_v2}))
    con.register('_fp_tgt_v2', _pd_v2.DataFrame({'target_id': fp_tgt_ids_v2}))
    df_fp_v2 = con.execute("""
        SELECT tk.s1_id, tk.target_id,
               tk.s1_norm_name, tk.t_norm_name,
               tk.s1_norm_addr, tk.t_norm_addr,
               tk.s1_door, tk.t_door
        FROM labeled_best tk
        WHERE tk.s1_id    IN (SELECT s1_id    FROM _fp_s1_v2)
          AND tk.target_id IN (SELECT target_id FROM _fp_tgt_v2)
          AND tk.split = 'val' AND tk.label = 0
    """).df()
    for _, r in df_fp_v2.iterrows():
        ns  = _fuzz.token_sort_ratio(str(r.get('s1_norm_name','') or ''), str(r.get('t_norm_name','') or ''))
        as_ = _fuzz.token_sort_ratio(str(r.get('s1_norm_addr','') or ''), str(r.get('t_norm_addr','') or ''))
        sd  = str(r.get('s1_door','') or '')
        td  = str(r.get('t_door','') or '')
        sn  = str(r.get('s1_norm_name','') or '')
        if ns >= 85 and as_ < 40:                 fp_cats_v2['same_name_diff_address'] += 1
        elif as_ >= 80 and ns < 40:               fp_cats_v2['same_address_diff_business'] += 1
        elif sd and td and sd != td and ns >= 70: fp_cats_v2['address_number_conflict'] += 1
        elif len(sn.replace(' ','')) <= 5:        fp_cats_v2['abbreviation_collision'] += 1
        else:                                     fp_cats_v2['generic_name_collision'] += 1
    fp_total_v2 = max(len(df_fp_v2), 1)
    del df_fp_v2; gc.collect()
else:
    fp_total_v2 = 1

fp_df_v2 = _pd_v2.DataFrame([{'Category':k,'Count':v,'Pct':f'{v/fp_total_v2*100:.1f}%'}
                              for k,v in sorted(fp_cats_v2.items(), key=lambda x:-x[1])])
print('\nFalse Positive Categories (v2):')
display(fp_df_v2)

# ── FALSE NEGATIVE ANALYSIS ────────────────────────────────────────────────
fn_s1_ids_v2  = val_s1_ids_v2[fn_mask_v2].tolist()
fn_tgt_ids_v2 = val_tgt_ids_v2[fn_mask_v2].tolist()

fn_cats_v2 = defaultdict(int)
if fn_s1_ids_v2:
    con.register('_fn_s1_v2',  _pd_v2.DataFrame({'s1_id':  fn_s1_ids_v2}))
    con.register('_fn_tgt_v2', _pd_v2.DataFrame({'target_id': fn_tgt_ids_v2}))
    df_fn_v2 = con.execute("""
        SELECT tk.s1_id, tk.target_id,
               tk.s1_norm_name, tk.t_norm_name,
               tk.s1_norm_addr, tk.t_norm_addr,
               tk.s1_door, tk.t_door
        FROM labeled_best tk
        WHERE tk.s1_id    IN (SELECT s1_id    FROM _fn_s1_v2)
          AND tk.target_id IN (SELECT target_id FROM _fn_tgt_v2)
          AND tk.split = 'val' AND tk.label = 1
    """).df()
    for _, r in df_fn_v2.iterrows():
        ns  = _fuzz.token_sort_ratio(str(r.get('s1_norm_name','') or ''), str(r.get('t_norm_name','') or ''))
        as_ = _fuzz.token_sort_ratio(str(r.get('s1_norm_addr','') or ''), str(r.get('t_norm_addr','') or ''))
        sn  = str(r.get('s1_norm_name','') or '')
        tn  = str(r.get('t_norm_name','') or '')
        sa  = str(r.get('s1_norm_addr','') or '')
        ta  = str(r.get('t_norm_addr','') or '')
        if ns < 50 and as_ < 50:                  fn_cats_v2['both_name_addr_weak'] += 1
        elif ns < 60:                             fn_cats_v2['name_typo_or_variation'] += 1
        elif as_ < 60:                            fn_cats_v2['address_typo_or_missing'] += 1
        elif not sn or not tn:                    fn_cats_v2['missing_business_name'] += 1
        elif not sa or not ta:                    fn_cats_v2['missing_address'] += 1
        else:                                     fn_cats_v2['other_low_confidence'] += 1
    fn_total_v2 = max(len(df_fn_v2), 1)
    del df_fn_v2; gc.collect()
else:
    fn_total_v2 = 1

fn_df_v2 = _pd_v2.DataFrame([{'Category':k,'Count':v,'Pct':f'{v/fn_total_v2*100:.1f}%'}
                              for k,v in sorted(fn_cats_v2.items(), key=lambda x:-x[1])])
print('\nFalse Negative Categories (v2):')
display(fn_df_v2)

# ── BLOCKING MISSES ────────────────────────────────────────────────────────
# True matches that never appeared in the best candidate pool
print('\nBlocking miss analysis...')
blocking_miss_cnt = con.execute(f"""
    SELECT count(*) FROM (
        SELECT s.entity_id, unnest(string_split(s.matched_entity_ids,',')) as true_tgt
        FROM s1_split s
        WHERE s.split='val'
          AND s.matched_entity_ids IS NOT NULL
          AND trim(s.matched_entity_ids) != ''
    ) gt
    WHERE NOT EXISTS (
        SELECT 1 FROM {BEST_TOPK_TABLE} c
        WHERE c.s1_id = gt.entity_id AND c.target_id = gt.true_tgt
    )
""").fetchone()[0]
total_true_links = con.execute("""
    SELECT sum(match_count) FROM s1_split WHERE split='val' AND match_count > 0
""").fetchone()[0] or 1
blocking_miss_pct = blocking_miss_cnt / total_true_links * 100
print(f'  True links in val:          {total_true_links:,}')
print(f'  Blocking misses:            {blocking_miss_cnt:,}  ({blocking_miss_pct:.2f}%)')
print(f'  Blocking recall (val):      {100-blocking_miss_pct:.2f}%')

# Save error analysis
err_rows = []
for cat, cnt in fp_cats_v2.items():
    err_rows.append({'type':'FP','category':cat,'count':cnt,'pct':round(cnt/fp_total_v2*100,1)})
for cat, cnt in fn_cats_v2.items():
    err_rows.append({'type':'FN','category':cat,'count':cnt,'pct':round(cnt/fn_total_v2*100,1)})
err_rows.append({'type':'BLOCKING_MISS','category':'never_in_pool',
                 'count':blocking_miss_cnt,'pct':round(blocking_miss_pct,2)})
_pd_v2.DataFrame(err_rows).to_csv('/content/experiment_v2/error_analysis_v2.csv', index=False)
print('Saved: /content/experiment_v2/error_analysis_v2.csv')


### Section 27: Save v2 Artifacts


In [ ]:
import joblib, json as _json, os
import numpy as _np2

V2_DIR = '/content/experiment_v2'
os.makedirs(V2_DIR, exist_ok=True)

# Save XGBoost v2
xgb_model_v2.save_model(os.path.join(V2_DIR, 'xgboost_v2.json'))

# Save CatBoost
cb_model.save_model(os.path.join(V2_DIR, 'catboost_v2.cbm'))

# Save feature config v2
with open(os.path.join(V2_DIR, 'feature_config_v2.json'), 'w') as f:
    _json.dump({'feature_names': FEATURE_NAMES_V2, 'count': N_FEAT_V2}, f, indent=2)

# Save best model config
best_v2_meta = {
    'best_model': best_v2_name,
    'best_blocking': BEST_STRATEGY,
    'K': K_BEST,
    'threshold': best_v2_tau,
    'macro_f05': best_rows_v2[best_v2_name]['macro_f05'],
    'precision': best_rows_v2[best_v2_name]['precision'],
    'recall':    best_rows_v2[best_v2_name]['recall'],
    'n_val_s1':  best_rows_v2[best_v2_name]['n_val_s1_evaluated'],
    'baseline': {'model': 'XGBoost', 'blocking': '8-pass K=25',
                 'macro_f05': 0.8841, 'precision': 0.9816, 'recall': 0.7777, 'threshold': 0.80},
}
with open(os.path.join(V2_DIR, 'best_model_v2.json'), 'w') as f:
    _json.dump(best_v2_meta, f, indent=2)

# Save val predictions (first 10k rows)
import pandas as _pd_v2
_vp = _pd_v2.DataFrame({
    's1_id':      val_s1_ids_v2[:10000],
    'target_id':  val_tgt_ids_v2[:10000],
    'label':      y_val_v2[:10000],
    'pred_prob':  best_v2_probs[:10000],
    'pred_match': (best_v2_probs[:10000] >= best_v2_tau).astype(_np2.int32),
})
_vp.to_csv(os.path.join(V2_DIR, 'validation_predictions_v2.csv'), index=False)
del _vp

print(f'v2 artifacts saved to {V2_DIR}/')
print(f'  xgboost_v2.json')
print(f'  catboost_v2.cbm')
print(f'  feature_config_v2.json')
print(f'  best_model_v2.json')
print(f'  blocking_comparison.csv')
print(f'  model_comparison_v2.csv')
print(f'  threshold_results_v2.csv')
print(f'  validation_predictions_v2.csv')
print(f'  error_analysis_v2.csv')


### Section 28: Final Decision Summary

Objective summary of all measured results. No claims about leaderboard performance.


In [ ]:
import pandas as _pd_v2

print('=' * 80)
print('EXPERIMENT SUMMARY')
print('=' * 80)

print()
print('CURRENT BASELINE:')
print(f'  Model:       XGBoost + 8-pass blocking K=25')
print(f'  Macro F0.5 = 0.8841  (known result)')
print(f'  Precision  = 0.9816')
print(f'  Recall     = 0.7777')
print(f'  Threshold  = 0.80')
print(f'  Val S1     = 30,001')

print()
print('NEW RESULTS:')
print(f'  Best blocking strategy: {BEST_STRATEGY}')
print(f'  Candidate recall@{K_BEST}:   {best_block_recall:.4f}')
print(f'  Baseline recall@25:    {blocking_results[0]["recall_at_25"]:.4f}')
recall_delta = best_block_recall - blocking_results[0]['recall_at_25']
print(f'  Recall delta:          {recall_delta:+.4f}')

print()
print('MODEL RESULTS (v2):')
for mname, _ in models_v2:
    br = best_rows_v2[mname]
    print(f'  {mname:<30}  F0.5={br["macro_f05"]:.4f}  P={br["precision"]:.4f}  R={br["recall"]:.4f}  tau={br["threshold"]:.2f}')

best_v2_f05 = best_rows_v2[best_v2_name]['macro_f05']
f05_delta = best_v2_f05 - 0.8841

print()
print('ANSWERS TO KEY QUESTIONS:')
print(f'  1. Did candidate recall improve?')
print(f'     Baseline recall@25 = {blocking_results[0]["recall_at_25"]:.4f}')
print(f'     Best strategy recall@{K_BEST} = {best_block_recall:.4f}  (delta={recall_delta:+.4f})')
print(f'     Answer: {"YES" if recall_delta > 0.001 else "MARGINAL" if recall_delta > 0 else "NO"}')

print(f'  2. Did Macro F0.5 improve?')
print(f'     Baseline = 0.8841  Best v2 = {best_v2_f05:.4f}  (delta={f05_delta:+.4f})')
print(f'     Answer: {"YES" if f05_delta > 0.001 else "MARGINAL" if f05_delta > 0 else "NO"}')

print(f'  3. Which model performed best?')
print(f'     {best_v2_name}  (Macro F0.5 = {best_v2_f05:.4f})')

print(f'  4. What threshold was optimal?')
print(f'     tau* = {best_v2_tau:.2f}')

print(f'  5. How much improvement occurred?')
print(f'     F0.5 delta = {f05_delta:+.4f}  ({f05_delta/0.8841*100:+.2f}% relative)')

print(f'  6. Was the improvement empirically meaningful?')
print(f'     Evaluated on {best_rows_v2[best_v2_name]["n_val_s1_evaluated"]:,} val S1 entities.')
print(f'     {"Improvement is meaningful (>0.005 F0.5)." if f05_delta > 0.005 else "Improvement is marginal (<0.005 F0.5). Consider further tuning." if f05_delta > 0 else "No improvement. Baseline remains best."}')

print(f'  7. What should we use for final production pipeline?')
if f05_delta > 0.005:
    print(f'     USE: {best_v2_name} + {BEST_STRATEGY} blocking at tau*={best_v2_tau:.2f}')
    print(f'     Baseline is superseded by v2 experiment.')
else:
    print(f'     KEEP: Baseline XGBoost + 8-pass K=25 at tau*=0.80')
    print(f'     v2 improvement is insufficient to justify added complexity.')

print()
print('NOTE: These are VALIDATION results only.')
print('      Do NOT proceed to full 1.73M test inference until pipeline is finalized.')
print('=' * 80)
